# ARC-AGI-2: conceptual lattice + RDR over ARCGraph objects (symbolic, no neural model)
Frozen prior vocabulary `fill_interior,ray,connect,border,bbox_fill,outline8,L_inertia,D12,S_holes,D4,D1,S_size,P_pixel,P_frame,D10,S_atoms,G_dsl`. Per task, RDR decision lists over lattice concepts are induced
from the training pairs only and executed on the test inputs. Unresolved slots get a 1x1 placeholder,
so any non-zero score comes from the induced rules.


In [ ]:
import hashlib, json, subprocess, sys, zipfile, os
from pathlib import Path
BUNDLE_SHA = '9eedcc404d6015a2c84db58c63ec97c3e55f282faaed5fb9eb479afe0a6ab4fb'
bundles = [p for p in Path('/kaggle/input').rglob('candidate.bundle')]
assert len(bundles) == 1, bundles
payload = bundles[0].parent
assert hashlib.sha256(bundles[0].read_bytes()).hexdigest() == BUNDLE_SHA
cand = Path('/kaggle/working/cand')
if not cand.exists():
    with zipfile.ZipFile(bundles[0]) as z:
        z.extractall(cand)
deps = Path('/kaggle/working/deps')
try:
    import rdflib  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--target', str(deps),
                    str(payload / 'rdflib-7.6.0-py3-none-any.whl')], check=True)
challenges = [p for p in Path('/kaggle/input').rglob('arc-agi_test_challenges.json')]
assert len(challenges) == 1, challenges
Path('/kaggle/working/probe').mkdir(exist_ok=True)
CODEX_SHA = '62d4c71378e546093b874b9a1a1ffc61758b85cd531c06b5d4ecab33df18693b'
codex = [p for p in Path('/kaggle/input').rglob('codex_atoms.tgz')]
if codex:
    import tarfile
    assert hashlib.sha256(codex[0].read_bytes()).hexdigest() == CODEX_SHA, 'codex atom bundle digest mismatch'
    cx = Path('/kaggle/working/codex')
    if not cx.exists():
        with tarfile.open(codex[0]) as t: t.extractall(cx)
    os.environ['CODEX_ROOT'] = str(cx); os.environ['S0_REPS'] = str(cx / 's0_reps.json')
print(payload, challenges[0], 'codex' if codex else 'no-codex')


In [ ]:
%%writefile /kaggle/working/probe/occupancy2.py
"""M1 occupancy probe: prior-concept layer + per-task conceptual lattice + RDR decision list.

Segmentation comes from registered ARCGraph abstractions. Attributes come only from a
fixed, task-independent prior layer (topology, geometry, number/extremes, property
chains). Rules are lattice concepts (extent = closure of a minimal generator of <= 2
attributes) arranged as an RDR-style decision list. No task identifiers; outputs are
used only on training pairs.
"""
from __future__ import annotations

import copy, json, os, signal, sys, time
from collections import Counter
from itertools import combinations
from types import SimpleNamespace

sys.path[:0] = ["/home/claude/work/stubs", "/home/claude/work/cand2"]
from image import Image  # noqa: E402

ABSTRACTIONS = ("nbccg", "ccgbr", "nbvcg", "nbhcg", "mcccg")
DIRS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
MAX_RULES = 5


class Timeout(BaseException):
    pass


# ---------------------------------------------------------------- segmentation (ARCGraph)
def segment(grid, abstraction):
    image = Image(SimpleNamespace(stop_search=False, check_time_limit=lambda: False,
                                  max_abstract_nodes=64), grid=copy.deepcopy(grid), name="m1")
    graph = getattr(image, Image.abstraction_ops[abstraction])()
    nodes = []
    for _, d in graph.graph.nodes(data=True):
        pix = [tuple(p) for p in d["nodes"]]
        colors = {p: grid[p[0]][p[1]] for p in pix}
        vals = set(colors.values())
        nodes.append({"pix": frozenset(pix), "colors": colors,
                      "color": next(iter(vals)) if len(vals) == 1 else None})
    if not nodes or len(nodes) > 64:
        raise ValueError("node_count")
    return nodes


def background(grid):
    return Counter(c for row in grid for c in row).most_common(1)[0][0]


# ---------------------------------------------------------------- prior layer
def norm_shape(pix):
    r0 = min(r for r, _ in pix); c0 = min(c for _, c in pix)
    return frozenset((r - r0, c - c0) for r, c in pix)


def bbox(pix):
    rs = [r for r, _ in pix]; cs = [c for _, c in pix]
    return min(rs), min(cs), max(rs), max(cs)


def has_hole(pix):
    r0, c0, r1, c1 = bbox(pix)
    if r1 - r0 < 2 or c1 - c0 < 2:
        return False
    free = {(r, c) for r in range(r0 - 1, r1 + 2) for c in range(c0 - 1, c1 + 2)} - set(pix)
    stack = [(r0 - 1, c0 - 1)]; seen = set(stack)
    while stack:
        r, c = stack.pop()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            q = (r + dr, c + dc)
            if q in free and q not in seen:
                seen.add(q); stack.append(q)
    return len(seen) < len(free)


def interior(pix):
    """Cells enclosed by pix (not reachable from outside its bbox)."""
    r0, c0, r1, c1 = bbox(pix)
    free = {(r, c) for r in range(r0 - 1, r1 + 2) for c in range(c0 - 1, c1 + 2)} - set(pix)
    stack = [(r0 - 1, c0 - 1)]; seen = set(stack)
    while stack:
        r, c = stack.pop()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            q = (r + dr, c + dc)
            if q in free and q not in seen:
                seen.add(q); stack.append(q)
    return free - seen


def attributes(nodes, grid):
    """Return per-node attribute sets plus chain-valued color sources."""
    H, W = len(grid), len(grid[0])
    sizes = [len(n["pix"]) for n in nodes]
    shapes = [norm_shape(n["pix"]) for n in nodes]
    colors = [n["color"] for n in nodes]
    ccount = Counter(c for c in colors if c is not None)
    scount = Counter(shapes)
    boxes = [bbox(n["pix"]) for n in nodes]
    inner = [interior(n["pix"]) for n in nodes]
    owner = {}
    for i, n in enumerate(nodes):
        for p in n["pix"]:
            owner[p] = i
    out, sources = [], []
    for i, n in enumerate(nodes):
        a = set(); src = {}
        pix = n["pix"]; r0, c0, r1, c1 = boxes[i]; h, w = r1 - r0 + 1, c1 - c0 + 1
        c = colors[i]
        a.add(f"color={c}" if c is not None else "multicolor")
        a.add(f"size={sizes[i]}" if sizes[i] <= 12 else "size>12")
        if sizes[i] == max(sizes): a.add("largest")
        if sizes[i] == min(sizes): a.add("smallest")
        if sizes.count(sizes[i]) == 1: a.add("size_unique")
        if sizes[i] == 1: a.add("single_pixel")
        if h == 1 and w > 1: a.add("hline")
        if w == 1 and h > 1: a.add("vline")
        if len(pix) == h * w and h * w > 1: a.add("filled_rect")
        if h == w: a.add("square_bbox")
        if len(pix) == 2 * (h + w) - 4 and h > 2 and w > 2 and has_hole(pix): a.add("hollow_rect")
        if inner[i]: a.add("has_hole")
        s = shapes[i]
        if s == frozenset((r, w - 1 - cc) for r, cc in s): a.add("sym_lr")
        if s == frozenset((h - 1 - r, cc) for r, cc in s): a.add("sym_ud")
        a.add("shape_unique" if scount[s] == 1 else "shape_shared")
        if c is not None:
            if ccount[c] == 1: a.add("color_unique")
            if ccount[c] == max(ccount.values()): a.add("color_most_common")
            if ccount[c] == min(ccount.values()): a.add("color_least_common")
        if r0 == 0 or c0 == 0 or r1 == H - 1 or c1 == W - 1: a.add("touches_border")
        if r0 == min(b[0] for b in boxes): a.add("topmost")
        if r1 == max(b[2] for b in boxes): a.add("bottommost")
        if c0 == min(b[1] for b in boxes): a.add("leftmost")
        if c1 == max(b[3] for b in boxes): a.add("rightmost")
        # topology: touching / containment (RCC-style) and property chains
        touch = set()
        for (r, cc) in pix:
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (1, -1), (-1, 1), (-1, -1)):
                j = owner.get((r + dr, cc + dc))
                if j is not None and j != i:
                    touch.add(j)
        a.add("touches_other" if touch else "isolated")
        tcols = {colors[j] for j in touch if colors[j] is not None}
        for tc in tcols:
            a.add(f"adjColor={tc}")
        if len(tcols) == 1: src["adj"] = next(iter(tcols))
        containers = [j for j in range(len(nodes)) if j != i and pix <= inner[j]]
        if containers:
            a.add("inside_other")
            j = min(containers, key=lambda j: sizes[j])
            if colors[j] is not None:
                a.add(f"contextColor={colors[j]}"); src["ctx"] = colors[j]
        if any(nodes[j]["pix"] <= inner[i] for j in range(len(nodes)) if j != i): a.add("contains_other")
        aligned = {colors[j] for j in range(len(nodes)) if j != i and colors[j] is not None and
                   (not (boxes[j][2] < r0 or boxes[j][0] > r1) or not (boxes[j][3] < c0 or boxes[j][1] > c1))}
        for ac in aligned:
            a.add(f"alignedColor={ac}")
        if len(aligned) == 1: src["aligned"] = next(iter(aligned))
        big = max(range(len(nodes)), key=lambda j: sizes[j])
        small = min(range(len(nodes)), key=lambda j: sizes[j])
        if sizes.count(sizes[big]) == 1 and colors[big] is not None: src["largest"] = colors[big]
        if sizes.count(sizes[small]) == 1 and colors[small] is not None: src["smallest"] = colors[small]
        if ccount:
            least = [k for k, v in ccount.items() if v == min(ccount.values())]
            if len(least) == 1: src["least_common"] = least[0]
        same_shape_other = [j for j in range(len(nodes)) if j != i and shapes[j] == s and colors[j] is not None]
        if same_shape_other and len({colors[j] for j in same_shape_other}) == 1:
            src["same_shape"] = colors[same_shape_other[0]]
        out.append(a); sources.append(src)
    return out, sources


# ---------------------------------------------------------------- labels (training only)
def slide_distance(node, grid, bg, direction, others):
    dr, dc = DIRS[direction]
    H, W = len(grid), len(grid[0]); k = 0
    while True:
        nxt = [(r + dr * (k + 1), c + dc * (k + 1)) for r, c in node["pix"]]
        if any(not (0 <= r < H and 0 <= c < W) for r, c in nxt): return k
        if any(p in others for p in nxt): return k
        k += 1
        if k > 30: return k


def candidate_labels(node, src, grid, out, bg, others):
    H, W = len(out), len(out[0])
    cols = node["colors"]
    now = {p: out[p[0]][p[1]] for p in node["pix"]}
    labs = set()
    if all(now[p] == cols[p] for p in node["pix"]):
        labs.add("keep")
    vals = set(now.values())
    if node["color"] is not None and len(vals) == 1:
        v = next(iter(vals))
        if v != node["color"]:
            if v == bg: labs.add("remove")
            labs.add(f"recolor:const:{v}")
            for name, sc in src.items():
                if sc == v: labs.add(f"recolor:{name}")
    # translations
    p0 = next(iter(node["pix"]))
    for r in range(H):
        for c in range(W):
            if out[r][c] != cols[p0]: continue
            d = (r - p0[0], c - p0[1])
            if d == (0, 0): continue
            if all(0 <= p[0] + d[0] < H and 0 <= p[1] + d[1] < W and
                   out[p[0] + d[0]][p[1] + d[1]] == cols[p] for p in node["pix"]):
                labs.add(f"move:{d[0]},{d[1]}")
                for name, (dr, dc) in DIRS.items():
                    k = slide_distance(node, grid, bg, name, others)
                    if k and (dr * k, dc * k) == d: labs.add(f"slide:{name}")
    return labs


def apply_labels(grid, nodes, srcs, labels, bg, others_list):
    canvas = [row[:] for row in grid]
    H, W = len(grid), len(grid[0])
    paints = []
    for n, s, lab, others in zip(nodes, srcs, labels, others_list):
        if lab in ("keep", "refine"): continue
        if lab == "remove":
            for p in n["pix"]: canvas[p[0]][p[1]] = bg
            continue
        if lab.startswith("recolor:"):
            _, kind, *rest = lab.split(":")
            v = int(rest[0]) if kind == "const" else s.get(kind)
            if v is None: return None
            paints += [(p, v) for p in n["pix"]]
            continue
        if lab.startswith("move:"):
            d = tuple(int(x) for x in lab[5:].split(","))
        elif lab.startswith("slide:"):
            k = slide_distance(n, grid, bg, lab[6:], others)
            d = (DIRS[lab[6:]][0] * k, DIRS[lab[6:]][1] * k)
        else:
            return None
        for p in n["pix"]: canvas[p[0]][p[1]] = bg
        for p in n["pix"]:
            q = (p[0] + d[0], p[1] + d[1])
            if not (0 <= q[0] < H and 0 <= q[1] < W): return None
            paints.append((q, n["colors"][p]))
    for (r, c), v in paints:
        canvas[r][c] = v
    return canvas



import os
VOCAB = set(filter(None, os.environ.get("M1B_VOCAB", "").split(",")))
EFFECTS = [e for e in ("fill_interior", "ray", "connect", "border", "bbox_fill", "outline8") if e in VOCAB]


def effect_cells(name, node, grid, bg, owner_color):
    H, W = len(grid), len(grid[0]); pix = node["pix"]
    if name == "fill_interior":
        return [{q for q in interior(pix) if 0 <= q[0] < H and 0 <= q[1] < W and grid[q[0]][q[1]] == bg}]
    if name == "bbox_fill":
        r0, c0, r1, c1 = bbox(pix)
        return [{(r, c) for r in range(r0, r1 + 1) for c in range(c0, c1 + 1) if grid[r][c] == bg}]
    if name == "border":
        r0, c0, r1, c1 = bbox(pix)
        return [{(r, c) for r in range(r0 - 1, r1 + 2) for c in range(c0 - 1, c1 + 2)
                 if 0 <= r < H and 0 <= c < W and not (r0 <= r <= r1 and c0 <= c <= c1) and grid[r][c] == bg}]
    if name == "outline8":
        hole = interior(pix)
        return [{(r + dr, c + dc) for r, c in pix for dr in (-1, 0, 1) for dc in (-1, 0, 1)
                 if 0 <= r + dr < H and 0 <= c + dc < W and (r + dr, c + dc) not in pix and (r + dr, c + dc) not in hole
                 and grid[r + dr][c + dc] == bg}]
    if name == "ray":
        outs = []
        for dr, dc in list(DIRS.values()) + [(1, 1), (1, -1), (-1, 1), (-1, -1)]:
            cells = set()
            for r, c in pix:
                q = (r + dr, c + dc)
                if q in pix: continue
                while 0 <= q[0] < H and 0 <= q[1] < W and grid[q[0]][q[1]] == bg:
                    cells.add(q); q = (q[0] + dr, q[1] + dc)
            outs.append(cells)
        return outs
    if name == "connect":
        cells = set()
        for r, c in pix:
            for dr, dc in DIRS.values():
                q = (r + dr, c + dc); walk = []
                if q in pix: continue
                while 0 <= q[0] < H and 0 <= q[1] < W and grid[q[0]][q[1]] == bg:
                    walk.append(q); q = (q[0] + dr, q[1] + dc)
                if walk and 0 <= q[0] < H and 0 <= q[1] < W and q not in pix and owner_color.get(q) == node["color"]:
                    cells.update(walk)
        return [cells]
    return []


def effect_options(node, src, grid, bg, owner_color, out=None, families=None):
    """All (effect, index, colorspec) options; with out given, keep those consistent with out."""
    opts = []
    for name in (families or EFFECTS):
        for idx, cells in enumerate(effect_cells(name, node, grid, bg, owner_color)):
            if not cells: continue
            if out is None:
                opts.append((f"{name}#{idx}", cells)); continue
            vals = {out[r][c] for r, c in cells}
            if len(vals) != 1: continue
            v = next(iter(vals))
            specs = [f"const:{v}"] + (["self"] if v == node["color"] else []) + [k for k, sv in src.items() if sv == v]
            opts.append((f"{name}#{idx}", cells, specs, v))
    return opts


def effect_labels(node, src, grid, out, bg, owner_color, changed, families=None):
    labs = set()
    for name, cells, specs, v in effect_options(node, src, grid, bg, owner_color, out, families):
        if cells & changed:
            labs |= {f"{name}@{s}" for s in specs}
    return labs or {"none"}


def apply_effects(canvas, grid, nodes, srcs, elabels, bg, owner_color):
    for n, s, lab in zip(nodes, srcs, elabels):
        if lab == "none": continue
        name, spec = lab.split("@")
        ename, idx = name.split("#")
        cells_list = effect_cells(ename, n, grid, bg, owner_color)
        idx = int(idx)
        if idx >= len(cells_list): return None
        if spec.startswith("const:"): v = int(spec[6:])
        elif spec == "self": v = n["color"]
        else: v = s.get(spec)
        if v is None: return None
        for r, c in cells_list[idx]: canvas[r][c] = v
    return canvas


# ---------------------------------------------------------------- DSL primitive: pixel refinement (cycle 3)
def px_attrs(node, grid, bg, nattrs):
    """Pixel-level concepts inside an object: position within the object's frame, boundary/interior,
    parity (checkerboard), neighbourhood colour, row/column alignment with outside colours."""
    H, W = len(grid), len(grid[0]); pix = node["pix"]
    r0, c0, r1, c1 = bbox(pix); h, w = r1 - r0 + 1, c1 - c0 + 1
    rowcols = {}; colcols = {}
    for r in range(H):
        for c in range(W):
            if (r, c) not in pix and grid[r][c] != bg:
                rowcols.setdefault(r, set()).add(grid[r][c]); colcols.setdefault(c, set()).add(grid[r][c])
    out = {}
    for (r, c) in pix:
        a = {"obj:" + x for x in nattrs}
        a.add(f"px:color={node['colors'][(r, c)]}")
        nb4 = [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]
        a.add("px:boundary" if any(q not in pix for q in nb4) else "px:interior")
        if (r in (r0, r1)) and (c in (c0, c1)): a.add("px:corner")
        if r == r0: a.add("px:top_edge")
        if r == r1: a.add("px:bottom_edge")
        if c == c0: a.add("px:left_edge")
        if c == c1: a.add("px:right_edge")
        a.add(f"px:row_parity={(r - r0) % 2}"); a.add(f"px:col_parity={(c - c0) % 2}")
        a.add(f"px:checker={(r - r0 + c - c0) % 2}")
        if h % 2 and r - r0 == h // 2: a.add("px:mid_row")
        if w % 2 and c - c0 == w // 2: a.add("px:mid_col")
        if r - r0 < 3: a.add(f"px:row_from_top={r - r0}")
        if r1 - r < 3: a.add(f"px:row_from_bottom={r1 - r}")
        if c - c0 < 3: a.add(f"px:col_from_left={c - c0}")
        if c1 - c < 3: a.add(f"px:col_from_right={c1 - c}")
        for q in nb4:
            if 0 <= q[0] < H and 0 <= q[1] < W and q not in pix: a.add(f"px:adj={grid[q[0]][q[1]]}")
        for v in rowcols.get(r, ()): a.add(f"px:row_has={v}")
        for v in colcols.get(c, ()): a.add(f"px:col_has={v}")
        out[(r, c)] = a
    return out


def px_labels(node, out_grid):
    res = {}
    for (r, c) in node["pix"]:
        v = out_grid[r][c]
        res[(r, c)] = {"keep"} if v == node["colors"][(r, c)] else {f"pc:{v}"}
    return res


def apply_pixel_rules(canvas, grid, nodes, at, labels, rp, bg):
    for n, a, lab in zip(nodes, at, labels):
        if lab != "refine": continue
        pa = px_attrs(n, grid, bg, a)
        pl = predict_labels_default(rp, [pa[p] for p in sorted(pa)], "keep")
        for p, l in zip(sorted(pa), pl):
            if l.startswith("pc:"): canvas[p[0]][p[1]] = int(l[3:])
    return canvas

# ---------------------------------------------------------------- lattice / RDR decision list

def clarified(cases):
    """FCA attribute clarification: drop attributes with identical extents or full extent; cap for pair search."""
    n = len(cases); full = (1 << n) - 1
    ext = {}
    for i, (at, _) in enumerate(cases):
        for a in at: ext[a] = ext.get(a, 0) | (1 << i)
    seen, attrs = set(), []
    for a in sorted(ext):
        e = ext[a]
        if e == full or e in seen: continue
        seen.add(e); attrs.append(a)
    pair_attrs = attrs if len(attrs) <= 150 else sorted(attrs, key=lambda a: -bin(ext[a]).count("1"))[:150]
    return attrs, ext, pair_attrs

def learn(cases, specific=False, mode="greedy"):
    if mode == "inertia":
        return learn_inertia(cases, specific)
    return learn_greedy(cases, specific)


def learn_inertia(cases, specific=False, default="keep"):
    """RDR with an inertia root: default conclusion = no change; exceptions are lattice concepts
    whose extent contains no case that must stay unchanged-incompatible with the exception label."""
    n = len(cases)
    if not any(default in ls for _, ls in cases): return None
    attrs, ext, pair_attrs = clarified(cases)
    labels = sorted({l for _, ls in cases for l in ls} - {default})
    valid = {l: sum(1 << i for i, (_, ls) in enumerate(cases) if l in ls) for l in labels}
    need = sum(1 << i for i, (_, ls) in enumerate(cases) if default not in ls)
    gens = [(a,) for a in sorted(ext)] + list(combinations(pair_attrs, 2))
    uncovered = (1 << n) - 1; rules = []
    while uncovered & need:
        if len(rules) >= MAX_RULES - 1: return None
        best = None
        for g in gens:
            e = uncovered
            for a in g: e &= ext[a]
            if not (e & need): continue
            full = (1 << n) - 1
            for a in g: full &= ext[a]
            gen = bin(full).count("1")
            for l in labels:
                if e & ~valid[l]: continue
                key = (bin(e & need).count("1"), (len(g) if specific else -len(g)), gen, tuple(g), l)
                if best is None or key > best[0]: best = (key, g, l, e)
        if best is None: return None
        _, g, l, e = best
        rules.append((g, l)); uncovered &= ~e
    return rules + [((), default)]


def learn_greedy(cases, specific=False):
    """cases: list of (attrset, validlabelset). Greedy decision list of lattice concepts."""
    n = len(cases)
    attrs, ext, pair_attrs = clarified(cases)
    labels = sorted({l for _, ls in cases for l in ls})
    valid = {l: sum(1 << i for i, (_, ls) in enumerate(cases) if l in ls) for l in labels}
    uncovered = (1 << n) - 1
    rules = []
    gens = [()] + [(a,) for a in sorted(ext)] + list(combinations(pair_attrs, 2))
    while uncovered:
        if len(rules) >= MAX_RULES: return None
        best = None
        for g in gens:
            e = uncovered
            for a in g: e &= ext[a]
            if not e: continue
            cov = bin(e).count("1")
            full = (1 << n) - 1
            for a in g: full &= ext[a]
            gen = bin(full).count("1")  # maximal concept subsumption: prefer the more general concept
            for l in labels:
                if e & ~valid[l]: continue
                key = (cov, (len(g) if specific else -len(g)), gen, l != "keep", tuple(g), l)
                if best is None or key > best[0]:
                    best = (key, g, l, e)
        if best is None: return None
        _, g, l, e = best
        rules.append((g, l)); uncovered &= ~e
    return rules


def predict_labels(rules, attrsets):
    labs = []
    for at in attrsets:
        for g, l in rules:
            if all(a in at for a in g):
                labs.append(l); break
        else:
            labs.append("keep")
    return labs


# ---------------------------------------------------------------- task driver

def n_holes(pix):
    cells = interior(pix); comps = 0; seen = set()
    for q in cells:
        if q in seen: continue
        comps += 1; stack = [q]; seen.add(q)
        while stack:
            r, c = stack.pop()
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                x = (r + dr, c + dc)
                if x in cells and x not in seen:
                    seen.add(x); stack.append(x)
    return comps

def extra_attrs(nodes, grid):
    H, W = len(grid), len(grid[0])
    sizes = [len(n["pix"]) for n in nodes]; colors = [n["color"] for n in nodes]
    shapes = [norm_shape(n["pix"]) for n in nodes]; boxes = [bbox(n["pix"]) for n in nodes]
    inner = [interior(n["pix"]) for n in nodes]
    ranks = sorted(set(sizes), reverse=True)
    owner = {p: i for i, n in enumerate(nodes) for p in n["pix"]}
    res = []
    for i, n in enumerate(nodes):
        a = set(); r0, c0, r1, c1 = boxes[i]
        a.add(f"D1:size_rank={ranks.index(sizes[i])}" if ranks.index(sizes[i]) < 4 else "D1:size_rank>3")
        a.add(f"D1:size_rank_asc={sorted(set(sizes)).index(sizes[i])}")
        same = [j for j in range(len(nodes)) if colors[j] == colors[i]]
        a.add(f"D2:n_same_color={min(len(same),5)}")
        for j in range(len(nodes)):
            if j != i and shapes[j] == shapes[i] and colors[j] is not None: a.add(f"D3:shape_as_color={colors[j]}")
        touch = {owner[(r+dr, c+dc)] for r, c in n["pix"] for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)) if owner.get((r+dr, c+dc), i) != i}
        a.add(f"D4:n_touch={min(len(touch),4)}")
        a.add("D5:top_half" if (r0+r1)/2 < H/2 else "D5:bottom_half"); a.add("D5:left_half" if (c0+c1)/2 < W/2 else "D5:right_half")
        a.add(f"D8:height={r1-r0+1}"); a.add(f"D8:width={c1-c0+1}")
        for j in range(len(nodes)):
            if j != i and nodes[j]["pix"] <= inner[i] and colors[j] is not None: a.add(f"D7:contains_color={colors[j]}")
            if j != i and colors[j] is not None and not (boxes[j][2] < r0 or boxes[j][0] > r1): a.add(f"D9:row_with_color={colors[j]}")
            if j != i and colors[j] is not None and not (boxes[j][3] < c0 or boxes[j][1] > c1): a.add(f"D9:col_with_color={colors[j]}")
        if sizes[i] == max(sizes[j] for j in same): a.add("D10:largest_of_its_color")
        if sizes[i] == min(sizes[j] for j in same): a.add("D10:smallest_of_its_color")
        a.add(f"D11:n_cells_of_color_in_bbox={sum(1 for r in range(r0,r1+1) for c in range(c0,c1+1) if grid[r][c]==colors[i])==sizes[i]}")
        a.add(f"D12:n_holes={min(n_holes(n['pix']), 5)}")
        big = max(range(len(nodes)), key=lambda j: (sizes[j], -j))
        if big != i:
            br0, bc0, br1, bc1 = boxes[big]
            cy, cx = (r0 + r1) / 2, (c0 + c1) / 2; by, bx = (br0 + br1) / 2, (bc0 + bc1) / 2
            a.add("D13:above_largest" if cy < by else ("D13:below_largest" if cy > by else "D13:level_with_largest"))
            a.add("D13:left_of_largest" if cx < bx else ("D13:right_of_largest" if cx > bx else "D13:centered_on_largest"))
            if not (r1 < br0 or r0 > br1): a.add("D13:row_overlaps_largest")
            if not (c1 < bc0 or c0 > bc1): a.add("D13:col_overlaps_largest")
        dens = sizes[i] / ((r1 - r0 + 1) * (c1 - c0 + 1))
        a.add("D14:dense_full" if dens == 1 else ("D14:dense_high" if dens >= 0.5 else "D14:dense_low"))
        res.append(a)
    return res



# ---------------------------------------------------------------- Codex selectors as concepts (cycle 5)
_CODEX = None
_CODEX_CACHE = {}
ROLE_KEYS = ("role", "marker", "source", "separator", "foreground", "base", "color", "background")


def _codex():
    global _CODEX
    if _CODEX is None:
        import importlib, json as _j
        root = os.environ.get("CODEX_ROOT", "/mnt/user-data/uploads/arc_extended_arga")
        if root not in sys.path: sys.path.insert(0, root)
        cr = importlib.import_module("wake.composition_runtime")
        spec = _j.load(open(os.environ.get("CODEX_SELECTORS", "/home/claude/work/au/seltime.json")))
        _CODEX = [(name, getattr(cr, name), v["spec"]) for name, v in sorted(spec.items())]
    return _CODEX


def _cells(v, H, W):
    if isinstance(v, (list, tuple, set)) and v:
        items = list(v)[:2000]
        if all(isinstance(x, (list, tuple)) and len(x) == 2 and all(isinstance(y, int) for y in x) for x in items):
            cs = {(x[0], x[1]) for x in items}
            if all(0 <= r < H and 0 <= c < W for r, c in cs): return cs
    return None


def codex_features(grid):
    """Run each Codex selector on the grid; return (role attributes by colour, cell-set attributes, chain sources)."""
    key = tuple(map(tuple, grid))
    if key in _CODEX_CACHE: return _CODEX_CACHE[key]
    H, W = len(grid), len(grid[0])
    roles, cellsets, sources = {}, {}, {}
    for name, f, spec in _codex():
        try:
            out = f(grid, spec)
        except Exception:
            continue
        if not isinstance(out, dict): continue
        short = name[len("_select_"):] if name.startswith("_select_") else name
        for k, v in out.items():
            tag = f"cx:{short}.{k}"
            if isinstance(v, int) and not isinstance(v, bool) and 0 <= v <= 9 and any(r in k for r in ROLE_KEYS):
                roles.setdefault(v, set()).add(tag); sources[tag] = v
            else:
                cs = _cells(v, H, W)
                if cs: cellsets[tag] = cs
    _CODEX_CACHE[key] = (roles, cellsets, sources)
    if len(_CODEX_CACHE) > 5000: _CODEX_CACHE.clear()
    return roles, cellsets, sources



# ---------------------------------------------------------------- S0 atoms (Codex detectors) as a fallback stratum (cycle 6)
_ATOMS = None
_ATOM_CACHE = {}
ATOM_ACTIVE = False


def _atoms():
    global _ATOMS
    if _ATOMS is None:
        import importlib, json as _j
        root = os.environ.get("CODEX_ROOT", "/home/claude/work/codex")
        if root not in sys.path: sys.path.insert(0, root)
        reps = _j.load(open(os.environ.get("S0_REPS", "/home/claude/work/s0/s0_reps.json")))
        if "S_atoms_used" in VOCAB or "S_atoms_primary" in VOCAB:
            used = set(_j.load(open(os.environ.get("S0_REPS_USED", "/home/claude/work/s0/s0_reps_used.json"))))
            reps = [r for r in reps if r in used]
        fns = []
        for name in sorted(reps):
            mod, fn = name.rsplit(".", 1)
            try:
                fns.append((name, getattr(importlib.import_module("detectors." + mod), fn)))
            except Exception:
                pass
        _ATOMS = fns
    return _ATOMS


def _collect_cells(ev, H, W, out):
    if isinstance(ev, dict):
        if isinstance(ev.get("row"), int) and isinstance(ev.get("col"), int):
            if 0 <= ev["row"] < H and 0 <= ev["col"] < W: out.add((ev["row"], ev["col"]))
        for v in ev.values(): _collect_cells(v, H, W, out)
    elif isinstance(ev, (list, tuple, set, frozenset)):
        if len(ev) == 2 and all(isinstance(x, int) and not isinstance(x, bool) for x in ev):
            r, c = ev
            if 0 <= r < H and 0 <= c < W: out.add((r, c))
            return
        for v in ev: _collect_cells(v, H, W, out)


def atom_cells(grid):
    key = tuple(map(tuple, grid))
    if key in _ATOM_CACHE: return _ATOM_CACHE[key]
    H, W = len(grid), len(grid[0]); res = {}
    for name, fn in _atoms():
        try:
            ev = fn([row[:] for row in grid])
        except Exception:
            continue
        if not ev: continue
        cs = set(); _collect_cells(ev, H, W, cs)
        if cs: res["at:" + name.split(".", 1)[1][:-len("_evidence")] if name.endswith("_evidence") else "at:" + name] = cs
    _ATOM_CACHE[key] = res
    if len(_ATOM_CACHE) > 4000: _ATOM_CACHE.clear()
    return res


def prepare(grid, abstraction):
    nodes = segment(grid, abstraction)
    bg = background(grid)
    at, src = attributes(nodes, grid)
    if "S_holes" in VOCAB or "S_size" in VOCAB:
        holes = [n_holes(n["pix"]) for n in nodes]
        Hg, Wg = len(grid), len(grid[0])
        spans = [bbox(n["pix"])[0] == 0 or bbox(n["pix"])[1] == 0 or bbox(n["pix"])[2] == Hg - 1 or bbox(n["pix"])[3] == Wg - 1
                 for n in nodes]  # frames/separators anchored to the border are not reference tokens
        for i, s in enumerate(src):
            if nodes[i]["color"] is None: continue
            for key, vals in (("same_holes", holes), ("same_size", [len(n["pix"]) for n in nodes])):
                if ("S_holes" if key == "same_holes" else "S_size") not in VOCAB: continue
                cand = [j for j in range(len(nodes)) if j != i and vals[j] == vals[i] and not spans[j]
                        and nodes[j]["color"] is not None and nodes[j]["color"] != nodes[i]["color"]]
                if not cand: continue
                size_i = len(nodes[i]["pix"])
                gap = {j: abs(len(nodes[j]["pix"]) - size_i) for j in cand}
                best = [j for j in cand if gap[j] == min(gap.values())]
                cs = {nodes[j]["color"] for j in best}
                if len(cs) == 1:
                    s[key] = next(iter(cs)); at[i].add(f"exists_{key}")  # OWL existential restriction
    if "S_codex" in VOCAB:
        roles, cellsets, csrc = codex_features(grid)
        for n, a, s in zip(nodes, at, src):
            if n["color"] is not None:
                a |= roles.get(n["color"], set())
            for tag, cs in cellsets.items():
                if n["pix"] & cs: a.add(tag)
            for tag, v in csrc.items():
                s[tag] = v
    if ATOM_ACTIVE or "S_atoms_primary" in VOCAB:
        for tag, cs in atom_cells(grid).items():
            for n, a in zip(nodes, at):
                if n["pix"] & cs: a.add(tag)
    dv = {v for v in VOCAB if v.startswith("D")}
    if dv:
        for a, e in zip(at, extra_attrs(nodes, grid)):
            a |= {x for x in e if x.split(":")[0] in dv}
    allpix = [set().union(*[m["pix"] for j, m in enumerate(nodes) if j != i]) if len(nodes) > 1 else set()
              for i in range(len(nodes))]
    owner_color = {p: n["colors"][p] for n in nodes for p in n["pix"]}
    return nodes, bg, at, src, allpix, owner_color


def predict_grid(grid, rules, abstraction):
    rb, re_ = rules[:2]
    rp = rules[2] if len(rules) > 2 else None
    nodes, bg, at, src, others, oc = prepare(grid, abstraction)
    blabels = predict_labels(rb, at)
    canvas = apply_labels(grid, nodes, src, blabels, bg, others)
    if canvas is None: return None
    if rp:
        canvas = apply_pixel_rules(canvas, grid, nodes, at, blabels, rp, bg)
    if re_:
        groups, cur = [], None
        for item in re_:
            if isinstance(item, tuple) and len(item) == 1 and str(item[0]).startswith("#"):
                cur = []; groups.append(cur)
            else:
                cur.append(item)
        for fr in groups:
            el = predict_labels_default(fr, at, "none")
            canvas = apply_effects(canvas, grid, nodes, src, el, bg, oc)
            if canvas is None: return None
    return canvas


def predict_labels_default(rules, attrsets, default):
    labs = []
    for at in attrsets:
        for g, l in rules:
            if all(a in at for a in g):
                labs.append(l); break
        else:
            labs.append(default)
    return labs


def fit(pairs, abstraction, specific=False, mode="greedy"):
    cases, preps = [], []
    for p in pairs:
        gi, go = p["input"], p["output"]
        if (len(gi), len(gi[0])) != (len(go), len(go[0])): return None, "extent_change"
        pr = prepare(gi, abstraction)
        nodes, bg, at, src, others, oc = pr
        labs = [candidate_labels(n, s, gi, go, bg, o) for n, s, o in zip(nodes, src, others)]
        if PIXEL_ACTIVE:
            labs = [l if l else {"refine"} for l in labs]
        if EFFECTS:
            labs = [l if l else {"keep"} for l in labs]  # own pixels may be repainted by an effect
        if any(not l for l in labs): return None, "unlabelled_node"
        cases += list(zip(at, labs)); preps.append((gi, go, pr))
    rb = learn(cases, specific, mode)
    if rb is None: return None, "not_occupied"
    rp = None
    if PIXEL_ACTIVE and any(l == "refine" for _, l in rb):
        pcases = []
        for gi, go, (nodes, bg, at, src, others, oc) in preps:
            for n, a, lab in zip(nodes, at, predict_labels(rb, at)):
                if lab != "refine": continue
                pa, pl = px_attrs(n, gi, bg, a), px_labels(n, go)
                pcases += [(pa[p], pl[p]) for p in sorted(pa)]
        if pcases:
            rp = learn(pcases, specific, mode)
            if rp is None: return None, "not_occupied"
    re_ = None
    if EFFECTS:
        changes, residual_any = [], False
        for gi, go, (nodes, bg, at, src, others, oc) in preps:
            canvas = predict_grid(gi, (rb, None, rp), abstraction)
            if canvas is None: return None, "not_representable"
            changed = {(r, c) for r in range(len(gi)) for c in range(len(gi[0])) if canvas[r][c] != go[r][c]}
            residual_any |= bool(changed); changes.append(changed)
        if residual_any:
            re_ = []
            remaining = list(EFFECTS)
            while remaining and any(changes):
                best = None
                for fam in remaining:
                    ecases = []
                    for (gi, go, (nodes, bg, at, src, others, oc)), changed in zip(preps, changes):
                        ecases += list(zip(at, [effect_labels(n, s, gi, go, bg, oc, changed, [fam]) for n, s in zip(nodes, src)]))
                    if all(ls == {"none"} for _, ls in ecases): continue
                    fr = learn(ecases, specific, mode)
                    if fr is None or all(l == "none" for _, l in fr): continue
                    # MDL: residual cells this family explains, per rule
                    newchanges, explained = [], 0
                    for (gi, go, (nodes, bg, at, src, others, oc)), changed in zip(preps, changes):
                        canvas = [row[:] for row in gi]
                        for r, c in changed: canvas[r][c] = None
                        el = predict_labels_default(fr, at, "none")
                        canvas = apply_effects(canvas, gi, nodes, src, el, bg, oc)
                        if canvas is None: explained = -1; break
                        still = {(r, c) for r, c in changed if canvas[r][c] != go[r][c]}
                        explained += len(changed) - len(still); newchanges.append(still)
                    if explained <= 0: continue
                    key = (explained / len(fr), explained, fam)
                    if best is None or key > best[0]: best = (key, fam, fr, newchanges)
                if best is None: break
                _, fam, fr, changes = best
                remaining.remove(fam)
                re_ += [("#" + fam,)] + fr
            re_ = re_ or None
    rules = (rb, re_, rp)
    for gi, go, _ in preps:
        if predict_grid(gi, rules, abstraction) != go:
            return None, "not_representable"
    return rules, "occupied"


def run_rules(grid, rules, abstraction):
    return predict_grid(grid, rules, abstraction)


def nrules(rules):
    rb, re_ = rules[:2]
    rp = rules[2] if len(rules) > 2 else None
    return len(rb) + (sum(1 for x in re_ if not (len(x) == 1)) if re_ else 0) + (len(rp) if rp else 0)



# ---------------------------------------------------------------- DSL primitive: frame (cycle 3)
def frame_candidates(task, limit=3):
    """Select, by a lattice concept, the unique object whose bounding box becomes the output frame."""
    outs = []
    for ab in ("nbccg", "mcccg", "ccgbr"):
        try:
            per_pair = []
            for p in task["train"]:
                nodes, bg, at, src, others, oc = prepare(p["input"], ab)
                h, w = len(p["output"]), len(p["output"][0])
                dims = [(bbox(n["pix"])[2] - bbox(n["pix"])[0] + 1, bbox(n["pix"])[3] - bbox(n["pix"])[1] + 1) for n in nodes]
                per_pair.append((nodes, at, [d == (h, w) for d in dims]))
            tests = [prepare(t["input"], ab) for t in task["test"]]
        except Timeout:
            raise
        except Exception:
            continue
        if not all(any(c) for _, _, c in per_pair): continue
        attrs = sorted({x for _, at, _ in per_pair for a in at for x in a})
        gens = [(a,) for a in attrs] + list(combinations(attrs, 2))
        for g in gens:
            ok = True
            for nodes, at, cand in per_pair:
                hits = [i for i, a in enumerate(at) if all(x in a for x in g)]
                if len(hits) != 1 or not cand[hits[0]]: ok = False; break
            if not ok: continue
            thits = [[i for i, a in enumerate(tp[2]) if all(x in a for x in g)] for tp in tests]
            if any(len(h) != 1 for h in thits): continue
            sel = [nodes[[i for i, a in enumerate(at) if all(x in a for x in g)][0]] for nodes, at, _ in per_pair]
            tsel = [tp[0][h[0]] for tp, h in zip(tests, thits)]
            outs.append((ab, g, sel, tsel))
            if len(outs) >= limit: return outs
    return outs


def crop(grid, node):
    r0, c0, r1, c1 = bbox(node["pix"])
    return [row[c0:c1 + 1] for row in grid[r0:r1 + 1]]


def solve_frame(task):
    global FRAME_ACTIVE
    found = []
    FRAME_ACTIVE = True
    try:
        for ab, g, sel, tsel in frame_candidates(task):
            inner = {"train": [{"input": crop(p["input"], n), "output": p["output"]} for p, n in zip(task["train"], sel)],
                     "test": [{"input": crop(t["input"], n)} for t, n in zip(task["test"], tsel)]}
            att, _, fnd = solve(inner)
            for f in att:
                found.append(dict(f, frame=(ab, g), rules=f["rules"]))
            if found: break
    finally:
        FRAME_ACTIVE = False
    return found

FRAME_ACTIVE = False

PIXEL_ACTIVE = False


def solve(task, abstractions=ABSTRACTIONS):
    """Simpler vocabulary first (MDL): pixel refinement engages only if no object-level program exists."""
    global PIXEL_ACTIVE
    PIXEL_ACTIVE = False
    res = solve_once(task, abstractions)
    if not res[2] and "P_pixel" in VOCAB:
        PIXEL_ACTIVE = True
        try:
            res = solve_once(task, abstractions)
        finally:
            PIXEL_ACTIVE = False
    if not res[2] and ("S_atoms" in VOCAB or "S_atoms_used" in VOCAB):
        global ATOM_ACTIVE
        ATOM_ACTIVE = True
        try:
            res = solve_once(task, abstractions)
            if not res[2] and "P_pixel" in VOCAB:
                PIXEL_ACTIVE = True
                try: res = solve_once(task, abstractions)
                finally: PIXEL_ACTIVE = False
        finally:
            ATOM_ACTIVE = False
    if not res[2] and "P_frame" in VOCAB and not FRAME_ACTIVE and any(
            (len(p["input"]), len(p["input"][0])) != (len(p["output"]), len(p["output"][0])) for p in task["train"]):
        fr = solve_frame(task)
        if fr:
            res = (fr[:2], dict(res[1], frame="occupied"), fr)
    if res[2] and len(res[0]) < 2 and "G_dsl" in VOCAB and not FRAME_ACTIVE:
        # attempt 2 (cycle 10): if the lattice has a single distinct prediction, the cheapest verified
        # whole-grid program with different predictions fills the unused second attempt
        import gdsl
        for p in gdsl.search(task):
            if p["preds"] != res[0][0]["preds"]:
                g2 = {"abstraction": "gdsl", "specific": False, "mode": "gdsl", "rules": ([], []), "program": p["program"], "preds": p["preds"]}
                res = (res[0] + [g2], res[1], res[2])
                break
    if not res[2] and "G_dsl" in VOCAB and not FRAME_ACTIVE:
        # whole-grid stratum (cycle 8): verified programs over grid primitives, engaged only when
        # the object lattice, pixel refinement, atoms and frame strata found no program
        import gdsl
        progs = gdsl.search(task)
        if progs:
            found = [{"abstraction": "gdsl", "specific": False, "mode": "gdsl", "rules": ([], []),
                      "program": p["program"], "preds": p["preds"]} for p in progs]
            res = (found[:2], dict(res[1], gdsl="occupied"), found)
    return res


def solve_once(task, abstractions=ABSTRACTIONS):
    found, statuses = [], {}
    for ab in abstractions:
        variants = ([(False, "inertia")] if "L_inertia" in VOCAB else []) + [(False, "greedy"), (True, "greedy")]
        for specific, mode in variants:
            try:
                rules, st = fit(task["train"], ab, specific, mode)
            except Timeout:
                raise
            except Exception as e:
                rules, st = None, f"error:{type(e).__name__}"
            statuses.setdefault(ab, st)
            if rules is None: continue
            try:
                preds = [run_rules(t["input"], rules, ab) for t in task["test"]]
            except Timeout:
                raise
            except Exception:
                continue
            if any(p is None for p in preds): continue
            found.append({"abstraction": ab, "specific": specific, "mode": mode, "rules": rules, "preds": preds})
    distinct = {json.dumps(f["preds"]) for f in found}
    if len(distinct) > 2:  # regeneration stability: concepts must survive leave-one-out
        for f in found:
            f["stable"] = bool(loo(task, [f]))
    attempts, seen = [], set()
    for f in sorted(found, key=lambda f: (not f.get("stable", True), nrules(f["rules"]))):
        key = json.dumps(f["preds"])
        if key not in seen:
            seen.add(key); attempts.append(f)
    return attempts[:2], statuses, found


def loo(task, found):
    if not found or len(task["train"]) < 2: return None
    f = found[0]
    for i in range(len(task["train"])):
        rest = task["train"][:i] + task["train"][i + 1:]
        try:
            rules, _ = fit(rest, f["abstraction"], f["specific"], f.get("mode", "greedy"))
            if rules is None or run_rules(task["train"][i]["input"], rules, f["abstraction"]) != task["train"][i]["output"]:
                return False
        except Timeout:
            raise
        except Exception:
            return False
    return True


def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--challenges"); ap.add_argument("--solutions"); ap.add_argument("--keys", default="")
    ap.add_argument("--out"); ap.add_argument("--seconds", type=int, default=60)
    args = ap.parse_args()
    ch = json.load(open(args.challenges)); so = json.load(open(args.solutions))
    keys = sorted(ch) if not args.keys else [k.strip() for k in open(args.keys) if k.strip()]
    done = set()
    try:
        done = {json.loads(l)["task"] for l in open(args.out)}
    except FileNotFoundError:
        pass
    signal.signal(signal.SIGALRM, lambda *a: (_ for _ in ()).throw(Timeout()))
    for k in keys:
        if k in done: continue
        row = {"task": k}; t0 = time.time()
        try:
            signal.alarm(args.seconds)
            attempts, statuses, found = solve(ch[k])
            row["statuses"] = statuses
            row["occupied"] = bool(found)
            row["n_programs"] = len(found)
            sol = so[k]
            row["test_exact"] = bool(attempts) and all(
                any(a["preds"][i] == sol[i] for a in attempts) for i in range(len(sol)))
            row["loo"] = loo(ch[k], found)
            if attempts:
                rb, re_ = attempts[0]["rules"][:2]
                rp = attempts[0]["rules"][2] if len(attempts[0]["rules"]) > 2 else None
                row["rules"] = [[list(g), l] for g, l in rb] + [[list(x[0]), "effect:" + x[1]] if len(x) == 2 else ["family", x[0]] for x in (re_ or [])] + [[list(g), "pixel:" + l] for g, l in (rp or [])]
                row["n_rules"] = nrules(attempts[0]["rules"])
                row["abstraction"] = attempts[0]["abstraction"]
                if "program" in attempts[0]: row["program"] = attempts[0]["program"]
        except Timeout:
            row["status"] = "timeout"
        finally:
            signal.alarm(0)
        row["seconds"] = round(time.time() - t0, 2)
        with open(args.out, "a") as f:
            f.write(json.dumps(row) + "\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/probe/codex_ops.py
"""Bind Codex's grid-level DSL (extended_transformations/*_grid_based) into the verified G-DSL search.

Parameter domains are read from each function's own source (string enums) and from the task's
training pairs (colours present, size ratios). Nothing task-specific: every value is either an
enum the function declares, a colour seen in the task's training grids, or a small integer.
"""
from __future__ import annotations
import glob, importlib, inspect, os, re, sys

_OPS = None
SKIP = {"fill_grid_based"}          # needs an object argument, not a grid-level op
ENUM_HINT = ("type", "axis", "corner", "mirror_grid", "combine_pattern", "concat_axis", "fill_direction",
             "shifting_direction", "direction", "connect_mode", "component_mode", "selection", "output_mode",
             "mirror_axis", "center_mode", "duplicate_mode", "split_axis", "fold", "overlay")
INTS = {"factor": (2, 3, 4), "grid_size": (2, 3, 4, 5), "output_width": (1, 2, 3), "degrees": (90, 180, 270),
        "duplicate": (2, 3), "duplicate_arbitrary": (2, 3), "connectivity": (4, 8)}


def _root():
    for r in (os.environ.get("CODEX_DSL_ROOT", ""), "/home/claude/work/cand2", "/kaggle/working/cand"):
        if r and os.path.isdir(os.path.join(r, "extended_transformations")): return r
    return None


def load():
    global _OPS
    if _OPS is not None: return _OPS
    _OPS = []
    root = _root()
    if not root: return _OPS
    for p in (root, os.path.join(root, "extended_transformations")):
        if p not in sys.path: sys.path.append(p)
    for f in sorted(glob.glob(os.path.join(root, "extended_transformations", "*_grid.py"))):
        name = os.path.basename(f)[:-3]
        try:
            mod = importlib.import_module("extended_transformations." + name)
            src = inspect.getsource(mod)
        except Exception:
            continue
        for fn in sorted(n for n in dir(mod) if n.endswith("_grid_based")):
            if fn in SKIP: continue
            f_ = getattr(mod, fn)
            try: sig = inspect.signature(f_)
            except Exception: continue
            params = []
            for pn, p in list(sig.parameters.items())[1:]:
                d = p.default if p.default is not inspect.Parameter.empty else None
                if pn == "classifier_params": continue
                kind, vals = "fixed", [d]
                if any(pn.endswith(h) or pn == h for h in ENUM_HINT) and (isinstance(d, str) or d is None):
                    vs = set(re.findall(rf'\b{pn}\s*==\s*["\']([^"\']+)["\']', src))
                    for grp in re.findall(rf'\b{pn}\s+in\s*[\(\[\{{]([^\)\]\}}]+)[\)\]\}}]', src):
                        vs |= set(re.findall(r'["\']([^"\']+)["\']', grp))
                    if vs: kind, vals = "enum", ([d] if d is not None else []) + sorted(vs - {d})
                elif "color" in pn:
                    kind = "colour"
                elif isinstance(d, bool):
                    kind, vals = "bool", [d, not d]
                elif pn in INTS:
                    kind, vals = "int", [d] + [v for v in INTS[pn] if v != d]
                params.append((pn, kind, vals, d))
            _OPS.append((fn, f_, params))
    return _OPS


_WL = None
def whitelist():
    global _WL
    if _WL is None:
        import json
        p = os.path.join(os.path.dirname(os.path.abspath(__file__)), "codex_types_general.json")
        _WL = {tuple(x) for x in json.load(open(p))["types"]} if os.path.exists(p) and os.environ.get("CODEX_ALL_TYPES") != "1" else None
    return _WL

def programs(train, per_op=70):
    """Yield (name, cost, fn) for Codex ops: enum type values x (default colours, then one colour varied)."""
    ops = load()
    if not ops: return
    cols = sorted({v for p in train[:1] for g in (p["input"], p["output"]) for r in g for v in r})
    for fn, f_, params in ops:
        n = 0
        enum_ps = [p for p in params if p[1] == "enum"]
        main = enum_ps[0] if enum_ps else None
        others = [p for p in params if p is not main]
        base = {pn: d for pn, _, _, d in params}
        wl = whitelist()
        for tv in (main[2] if main else [None]):
            if n > per_op: break
            if wl is not None and (fn.replace("_grid_based", ""), str(tv)) not in wl: continue
            kw = dict(base)
            if main: kw[main[0]] = tv
            variants = [dict(kw)]
            for pn, kind, vals, d in others:
                opts = cols if kind == "colour" else vals[1:] if kind in ("enum", "bool", "int") else []
                for v in opts:
                    if v == d: continue
                    k2 = dict(kw); k2[pn] = v; variants.append(k2)
            # two colour params varied together over induced colours (e.g. colour1 -> colour2)
            cps = [p for p in others if p[1] == "colour"]
            if len(cps) >= 2:
                for a in cols:
                    for b in cols:
                        if a != b:
                            k2 = dict(kw); k2[cps[0][0]] = a; k2[cps[1][0]] = b; variants.append(k2)
            for k2 in variants:
                n += 1
                if n > per_op: break
                nd = sum(1 for pn, _, _, d in params if k2.get(pn) != d and (not main or pn != main[0]))
                label = f"codex:{fn.replace('_grid_based', '')}[{tv}]" + ("" if not nd else "(" + ",".join(f"{pn}={k2[pn]}" for pn, _, _, d in params if k2.get(pn) != d and (not main or pn != main[0])) + ")")
                yield (label, 3 + nd, (lambda g, f_=f_, k2=k2: f_([list(r) for r in g], **k2)))


In [ ]:
%%writefile /kaggle/working/probe/codex_types_general.json
{
 "rule": "Codex grid-op types admitted only if they solve >=2 distinct training tasks (cycle 8)",
 "types": [
  [
   "beam",
   "box_based"
  ],
  [
   "connect",
   "connect_rectangles"
  ],
  [
   "downscale",
   "horizontal"
  ],
  [
   "extract",
   "periodic_hole_patch"
  ],
  [
   "mirror",
   "diagonal"
  ],
  [
   "placement",
   "align_all_colors_to_guide_top"
  ],
  [
   "tile",
   "macro_mask_self_substitution"
  ]
 ]
}

In [ ]:
%%writefile /kaggle/working/probe/fam_crop_by_score.py
"""crop.by_score: segment the input into units, pick the unit that is the arg-extremum of a score,
output a part of it (bbox / interior / bbox-minus-marker / enclosed content).

One parametrised primitive  crop-by-score[units, score, extremum, part]:
  units    : c4 | c8 (single-colour comps) | m4 | m8 (multi-colour comps) | k2 (clusters, gap<=1)
             | frames (rectangles with a complete one-colour border) | panels (separator lines) | halves
  score    : area | bbox-area | n-colours | minority | count[c] | n-comps[c] | n-shape[c,S]
             | near[c,k] (marker cells within k of the unit, unit itself not of colour c) | has[c]
  extremum : max | min (unique winner required) | all (union of every unit with has[c])
  part     : bbox | inner (bbox shrunk by 1) | nomark[c] (marker colour erased, bbox of rest) | content
All parameters (unit kind, colour c, shape S, distance k, extremum, part) are induced: a program is
only yielded when on every training pair the chosen unit's part equals the output.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox, crop

# ------------------------------------------------------------------ units
def _rect_cells(r0, c0, r1, c1):
    return [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)]

def _clusters(g, bg, gap):
    h, w = H(g), W(g); seen = set(); out = []
    fg = {(y, x) for y in range(h) for x in range(w) if g[y][x] != bg}
    d = gap + 1
    for s in fg:
        if s in seen: continue
        seen.add(s); st = [s]; cells = []
        while st:
            y, x = st.pop(); cells.append((y, x))
            for dy in range(-d, d + 1):
                for dx in range(-d, d + 1):
                    q = (y + dy, x + dx)
                    if q in fg and q not in seen: seen.add(q); st.append(q)
        out.append(cells)
    return out

def _frames(g, bg):
    """Rectangles >=3x3 whose whole border is one non-bg colour (noise elsewhere allowed); maximal only."""
    h, w = H(g), W(g)
    R = [[0] * (w + 1) for _ in range(h + 1)]; D = [[0] * (w + 1) for _ in range(h + 1)]
    for y in range(h - 1, -1, -1):
        for x in range(w - 1, -1, -1):
            v = g[y][x]
            R[y][x] = 1 + (R[y][x + 1] if x + 1 < w and g[y][x + 1] == v else 0)
            D[y][x] = 1 + (D[y + 1][x] if y + 1 < h and g[y + 1][x] == v else 0)
    rects = []
    for y in range(h):
        for x in range(w):
            if g[y][x] == bg or R[y][x] < 3 or D[y][x] < 3: continue
            for ww in range(R[y][x], 2, -1):
                for hh in range(D[y][x], 2, -1):
                    x1, y1 = x + ww - 1, y + hh - 1
                    if D[y][x1] >= hh and R[y1][x] >= ww:
                        rects.append((y, x, y1, x1))
    rects = [r for r in rects if not any(o != r and o[0] <= r[0] and o[1] <= r[1] and o[2] >= r[2] and o[3] >= r[3] and g[o[0]][o[1]] == g[r[0]][r[1]] for o in rects)]
    if len(rects) > 40: return []
    return [_rect_cells(*r) for r in rects]

def _panels(g):
    h, w = H(g), W(g)
    for sc in sorted({v for r in g for v in r}):
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols or len(rows) == h or len(cols) == w: continue
        rb = [-1] + rows + [h]; cb = [-1] + cols + [w]; out = []
        for a in range(len(rb) - 1):
            for b in range(len(cb) - 1):
                if rb[a + 1] - rb[a] > 1 and cb[b + 1] - cb[b] > 1:
                    out.append(_rect_cells(rb[a] + 1, cb[b] + 1, rb[a + 1] - 1, cb[b + 1] - 1))
        if len(out) >= 2: return out
    return []

def _halves(g):
    h, w = H(g), W(g); out = []
    if w % 2 == 0: out += [_rect_cells(0, 0, h - 1, w // 2 - 1), _rect_cells(0, w // 2, h - 1, w - 1)]
    if h % 2 == 0: out += [_rect_cells(0, 0, h // 2 - 1, w - 1), _rect_cells(h // 2, 0, h - 1, w - 1)]
    return out

UNITS = ('m4', 'm8', 'c4', 'c8', 'k1', 'frames', 'panels', 'halves')
RECT_UNITS = ('frames', 'panels', 'halves')

def border_bg(g):
    h, w = H(g), W(g)
    cells = [g[0][x] for x in range(w)] + [g[h - 1][x] for x in range(w)] + [g[y][0] for y in range(h)] + [g[y][w - 1] for y in range(h)]
    return Counter(cells).most_common(1)[0][0]

BGS = {'mode': bg_of, 'border': border_bg}

def units(g, kind, bg):
    if kind == 'c4': return objects(g, bg, False, True)
    if kind == 'c8': return objects(g, bg, True, True)
    if kind == 'm4': return objects(g, bg, False, False)
    if kind == 'm8': return objects(g, bg, True, False)
    if kind == 'k1': return _clusters(g, bg, 1)
    if kind == 'frames': return _frames(g, bg)
    if kind == 'panels': return _panels(g)
    if kind == 'halves': return _halves(g)
    return []

# ------------------------------------------------------------------ parts
def part(g, u, how, bg):
    if how == 'bbox': return crop(g, bbox(u))
    if how == 'inner':
        r0, c0, r1, c1 = bbox(u)
        if r1 - r0 < 2 or c1 - c0 < 2: return None
        return crop(g, (r0 + 1, c0 + 1, r1 - 1, c1 - 1))
    if how.startswith('nomark'):
        c = int(how[6:])
        if not any(g[y][x] == c for y, x in u): return None
        rest = [(y, x) for y, x in u if g[y][x] != c]
        if not rest: return None
        return [[bg if v == c else v for v in r] for r in crop(g, bbox(rest))]
    if how == 'content':
        r0, c0, r1, c1 = bbox(u); us = set(u)
        sub = [[g[y][x] if (y, x) in us else bg for x in range(W(g))] for y in range(H(g))]
        inner = [ob for ob in objects(sub, bg, False, True)
                 if all(r0 < y < r1 and c0 < x < c1 for y, x in ob)]
        cells = [c for ob in inner for c in ob]
        if not cells: return None
        return crop(g, bbox(cells))
    return None

# ------------------------------------------------------------------ scores
def _shape(cells):
    r0, c0, _, _ = bbox(cells); return tuple(sorted((y - r0, x - c0) for y, x in cells))

def score(g, u, s, bg):
    """Numeric score of unit u, or None when the unit is not a candidate for this score."""
    kind = s[0]
    if kind == 'area': return sum(g[y][x] != bg for y, x in u)
    if kind == 'barea':
        r0, c0, r1, c1 = bbox(u); return (r1 - r0 + 1) * (c1 - c0 + 1)
    vals = [g[y][x] for y, x in u if g[y][x] != bg]
    if kind == 'ncol': return len(set(vals))
    if kind == 'minority':
        if not vals: return 0
        return len(vals) - Counter(vals).most_common(1)[0][1]
    c = s[1]
    if kind == 'cnt': return sum(v == c for v in vals)
    if kind == 'has': return int(c in vals)
    if kind == 'ncomp':
        cells = [(y, x) for y, x in u if g[y][x] == c]
        if not cells: return 0
        sub = [[bg] * W(g) for _ in range(H(g))]
        for y, x in cells: sub[y][x] = c
        return len(objects(sub, bg, False, True))
    if kind == 'nshape':   # occurrences of template S (all cells colour c) inside the unit
        us = set(u); S = s[2]
        return sum(all((y + dy, x + dx) in us and g[y + dy][x + dx] == c for dy, dx in S) for y, x in u)
    if kind == 'near':
        if c in vals: return None
        k = s[2]; r0, c0, r1, c1 = bbox(u)
        return sum(g[y][x] == c for y in range(max(0, r0 - k), min(H(g), r1 + k + 1))
                   for x in range(max(0, c0 - k), min(W(g), c1 + k + 1)))
    return None

def choose(g, us, s, ext, bg, pt=None):
    sc = [score(g, u, s, bg) if (pt is None or ext == 'all' or part(g, u, pt, bg) is not None) else None for u in us]
    idx = [i for i, v in enumerate(sc) if v is not None]
    if len(idx) < 2: return None
    if ext == 'all':
        hits = [i for i in idx if sc[i] == 1]
        if not hits or len(hits) == len(idx): return None
        return [c for i in hits for c in us[i]]
    key = (lambda i: sc[i]) if ext == 'max' else (lambda i: -sc[i])
    best = max(key(i) for i in idx)
    hits = [i for i in idx if key(i) == best]
    return us[hits[0]] if len(hits) == 1 else None

def _sname(s):
    return s[0] if len(s) == 1 else s[0] + '[' + ','.join('S' if isinstance(a, tuple) else str(a) for a in s[1:]) + ']'

# ------------------------------------------------------------------ family
def _programs(train, bgm):
    bgf = BGS[bgm]
    o0 = train[0]['output']
    bgs = [bgf(p['input']) for p in train]
    cols = sorted(set.intersection(*[{v for r in p['input'] for v in r} - {b} for p, b in zip(train, bgs)]))
    parts = ['bbox', 'inner', 'content'] + ['nomark%d' % c for c in cols]
    # score library (colour / shape / distance parameters drawn from the training pairs)
    lib = [('area',), ('barea',), ('ncol',), ('minority',)]
    for c in cols:
        lib += [('cnt', c), ('ncomp', c), ('has', c)] + [('near', c, k) for k in (1, 2, 3)]
        # templates: multi-cell colour-c shapes present as components in every training output
        shp = None
        for p in train:
            o = p['output']; ob = bg_of(o)
            here = {_shape(q) for q in objects(o, ob, False, True) if o[q[0][0]][q[0][1]] == c and len(q) > 1}
            shp = here if shp is None else shp & here
        if shp and len(shp) <= 3: lib += [('nshape', c, S) for S in sorted(shp)]
    for uk in UNITS:
        per = []
        for p, bg in zip(train, bgs):
            g = p['input']; us = units(g, uk, bg)
            if len(us) < 2 or len(us) > 60: per = None; break
            per.append((g, us, bg, p['output']))
        if not per: continue
        for pt in parts:
            if pt == 'content' and uk != 'k1': continue
            match = [{i for i, u in enumerate(us) if part(g, u, pt, bg) == o} for g, us, bg, o in per]
            single_ok = all(match)
            for s in lib:
                exts = ('max', 'min') + (('all',) if s[0] == 'has' else ())
                for ext in exts:
                    if ext != 'all' and not single_ok: continue
                    ok = True
                    for (g, us, bg, o), m in zip(per, match):
                        ch = choose(g, us, s, ext, bg, pt)
                        if ch is None: ok = False; break
                        if ext == 'all':
                            if part(g, ch, pt, bg) != o: ok = False; break
                        elif not any(ch is us[i] for i in m): ok = False; break
                    if not ok: continue
                    def fn(g, uk=uk, pt=pt, s=s, ext=ext, bgf=bgf):
                        bg = bgf(g); us = units(g, uk, bg)
                        if len(us) < 2: return None
                        ch = choose(g, us, s, ext, bg, pt)
                        return part(g, ch, pt, bg) if ch else None
                    yield (f"crop-by-score[{uk},{_sname(s)},{ext},{pt}{',bg=mode' if bgm == 'mode' else ''}]", 4, fn)

def fam_crop_by_score(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if H(o0) > H(i0) or W(o0) > W(i0) or (H(o0), W(o0)) == (H(i0), W(i0)): return
    for p in train:
        if H(p['output']) > H(p['input']) or W(p['output']) > W(p['input']): return
    bgmodes = ['border'] + (['mode'] if any(border_bg(p['input']) != bg_of(p['input']) for p in train) else [])
    n = 0
    for bgm in bgmodes:
        for prog in _programs(train, bgm):
            yield prog; n += 1
            if n >= 12: return

FAMILIES = (fam_crop_by_score,)


In [ ]:
%%writefile /kaggle/working/probe/fam_fill_region_by_property.py
"""fill.region_by_property: segment the grid into regions separated by walls, then paint each whole
region by a rule on one region feature.  Everything is induced from the training pairs:
  region colour r  : the input colour of (most) changed cells
  membership mode  : 'col'  region cells = colour r
                     'wall' region cells = every colour except the dominant wall colour w (most frequent non-r colour)
                     'seed' region cells = r plus isolated single-cell seeds (8-isolated, size-1 objects)
  connectivity     : 4 | 8
  feature          : border (touches edge) | size-rank | is-rect | size parity | seed majority colour |
                     adjacent-marker majority colour | adjacent-has colour c ; optionally paired with border
  table            : feature value -> action, action in {keep, constant colour, W (dominant wall colour),
                     self (the feature value is itself a colour)}.  Unseen feature values refuse (None).
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))


def dominant_wall(g, r):
    c = Counter(v for row in g for v in row if v != r)
    return c.most_common(1)[0][0] if c else None


def member_mask(g, r, mode):
    h, w = H(g), W(g)
    if mode == 'col':
        return [[g[y][x] == r for x in range(w)] for y in range(h)]
    if mode == 'wall':
        wc = dominant_wall(g, r)
        return [[g[y][x] != wc for x in range(w)] for y in range(h)]
    m = [[g[y][x] == r for x in range(w)] for y in range(h)]
    for y in range(h):
        for x in range(w):
            v = g[y][x]
            if v == r: continue
            if all(not (0 <= y + dy < h and 0 <= x + dx < w) or g[y + dy][x + dx] != v for dy, dx in N8):
                m[y][x] = True
    return m


def regions(g, mask, diag):
    h, w = H(g), W(g); seen = [[False] * w for _ in range(h)]; out = []
    nb = N8 if diag else N4
    for y0 in range(h):
        for x0 in range(w):
            if seen[y0][x0] or not mask[y0][x0]: continue
            st = [(y0, x0)]; seen[y0][x0] = True; cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and mask[yy][xx]:
                        seen[yy][xx] = True; st.append((yy, xx))
            out.append(cells)
    return out


def neighbours(g, cells, mask):
    h, w = H(g), W(g); s = set(cells); nb = set()
    for y, x in cells:
        for dy, dx in N4:
            yy, xx = y + dy, x + dx
            if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in s and not mask[yy][xx]:
                nb.add((yy, xx))
    return nb


def majority(cnt):
    if not cnt: return None
    mc = cnt.most_common(2)
    if len(mc) > 1 and mc[0][1] == mc[1][1]: return 'tie'
    return mc[0][0]


def feature_values(g, regs, mask, r, feat):
    h, w = H(g), W(g)
    if feat == 'border':
        return [any(y in (0, h - 1) or x in (0, w - 1) for y, x in c) for c in regs]
    if feat == 'size_rank':
        sizes = sorted({len(c) for c in regs}, reverse=True)
        def rk(n):
            if len(sizes) == 1: return 'only'
            if n == sizes[0]: return 'max'
            if n == sizes[-1]: return 'min'
            return '2nd' if n == sizes[1] else 'mid'
        return [rk(len(c)) for c in regs]
    if feat == 'is_rect':
        out = []
        for c in regs:
            ys = [y for y, _ in c]; xs = [x for _, x in c]
            out.append(len(c) == (max(ys) - min(ys) + 1) * (max(xs) - min(xs) + 1))
        return out
    if feat == 'parity':
        return [len(c) % 2 for c in regs]
    if feat == 'seedmaj':
        return [majority(Counter(g[y][x] for y, x in c if g[y][x] != r)) for c in regs]
    wc = dominant_wall(g, r)
    if feat == 'adjmaj':
        return [majority(Counter(g[y][x] for y, x in neighbours(g, c, mask) if g[y][x] != wc)) for c in regs]
    if feat.startswith('adjhas'):
        col = int(feat[6:])
        return [any(g[y][x] == col for y, x in neighbours(g, c, mask)) or any(g[y][x] == col for y, x in c) for c in regs]
    if feat == 'seedrank':   # region holding the most / fewest foreign (non-region-colour) cells
        cnt = [sum(g[y][x] != r for y, x in c) for c in regs]
        mx, mn = max(cnt), min(cnt)
        if mx == mn: return ['only'] * len(regs)
        return ['max' if k == mx and cnt.count(mx) == 1 else 'min' if k == mn else 'mid' for k in cnt]
    raise ValueError(feat)


SELF_FEATS = ('seedmaj', 'adjmaj')


def region_features(g, r, mode, diag, feat, with_border, scope='all'):
    if r == 'bg': r = bg_of(g)
    mask = member_mask(g, r, mode)
    regs = regions(g, mask, diag)
    if not regs or len(regs) > 200: return None
    vals = feature_values(g, regs, mask, r, feat)
    if with_border:
        bs = feature_values(g, regs, mask, r, 'border')
        vals = [(b, v) for b, v in zip(bs, vals)]
    if scope == 'r':   # only the region-colour cells are repainted; seeds/noise inside stay
        regs = [[(y, x) for y, x in c if g[y][x] == r] for c in regs]
    return regs, vals


def base_val(v, with_border):
    return v[1] if with_border else v


def is_col(v):
    return isinstance(v, int) and not isinstance(v, bool)


def skey(v, with_border):
    """In self mode every colour-valued feature shares one table key (its action is 'self' or a constant)."""
    bv = base_val(v, with_border)
    if not is_col(bv): return v
    return (v[0], '*') if with_border else '*'


def candidate_actions(gi, go, cells, wc):
    acts = set()
    if all(gi[y][x] == go[y][x] for y, x in cells): acts.add('keep')
    cs = {go[y][x] for y, x in cells}
    if len(cs) == 1:
        c = cs.pop(); acts.add(c)
        if c == wc: acts.add('W')
    return acts


def induce(train, r, mode, diag, feat, with_border, selfmode, scope):
    table = {}; any_change = False
    for p in train:
        gi, go = p['input'], p['output']
        rf = region_features(gi, r, mode, diag, feat, with_border, scope)
        if rf is None: return None
        regs, vals = rf
        cover = set()
        wc = dominant_wall(gi, bg_of(gi) if r == 'bg' else r)
        for c, v in zip(regs, vals):
            if not c: continue
            cover.update(c)
            acts = candidate_actions(gi, go, c, wc)
            if not acts: return None
            if selfmode:
                bv = base_val(v, with_border)
                if is_col(bv) and bv in acts: acts = acts | {'self'}
                v = skey(v, with_border)
            prev = table.get(v)
            table[v] = acts if prev is None else prev & acts
            if not table[v]: return None
        # every changed cell must lie in some region
        for y in range(H(gi)):
            for x in range(W(gi)):
                if gi[y][x] != go[y][x] and (y, x) not in cover: return None
    fixed = {}
    for v, acts in table.items():
        # prefer keep; else self; else the single consistent constant colour; else the dominant-wall-colour role
        ints = [k for k in acts if isinstance(k, int)]
        a = 'keep' if 'keep' in acts else 'self' if 'self' in acts else ints[0] if len(ints) == 1 else 'W'
        fixed[v] = a
        if a != 'keep': any_change = True
    if not any_change: return None
    return fixed


def apply(g, r, mode, diag, feat, with_border, selfmode, scope, table):
    rf = region_features(g, r, mode, diag, feat, with_border, scope)
    if rf is None: return None
    regs, vals = rf
    wc = dominant_wall(g, bg_of(g) if r == 'bg' else r)
    out = [row[:] for row in g]
    for c, v in zip(regs, vals):
        bv = base_val(v, with_border)
        k = skey(v, with_border) if selfmode else v
        if k not in table: return None
        a = table[k]
        if a == 'keep': continue
        col = wc if a == 'W' else bv if a == 'self' else a
        if not is_col(col): return None
        for y, x in c: out[y][x] = col
    return out


def fam_region_by_property(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    ch = Counter()
    for p in train:
        gi, go = p['input'], p['output']
        for y in range(H(gi)):
            for x in range(W(gi)):
                if gi[y][x] != go[y][x]: ch[gi[y][x]] += 1
    if not ch: return
    rs = [ch.most_common(1)[0][0]]
    if bg_of(i0) not in rs: rs.append(bg_of(i0))
    if len({bg_of(p['input']) for p in train}) > 1: rs.append('bg')   # region colour as a per-grid role
    # colours relevant for adjhas: colours painted in outputs
    painted = set()
    for p in train:
        gi, go = p['input'], p['output']
        painted |= {go[y][x] for y in range(H(gi)) for x in range(W(gi)) if gi[y][x] != go[y][x]}
    feats = ['border', 'size_rank', 'is_rect', 'parity', 'seedrank', 'seedmaj', 'adjmaj'] + ['adjhas%d' % c for c in sorted(painted)][:3]
    from itertools import product
    n = 0
    for r, mode, diag, feat in product(rs, ('col', 'seed', 'wall'), (False, True), feats):
        if mode == 'col' and feat == 'seedmaj': continue
        sm = feat in SELF_FEATS
        for wb, scope in product((False,) if feat == 'border' else (False, True), ('all',) if mode == 'col' else ('all', 'r')):
            t = induce(train, r, mode, diag, feat, wb, sm, scope)
            if t is None: continue
            n += 1
            if n > 40: return
            name = f"region-fill:{feat}{'+border' if wb else ''}[r{r},{mode},{'8' if diag else '4'}{',r-only' if scope == 'r' else ''}]"
            yield (name, 4 + wb, lambda g, a=(r, mode, diag, feat, wb, sm, scope, t): apply(g, *a))

FAMILIES = (fam_region_by_property,)


In [ ]:
%%writefile /kaggle/working/probe/fam_lines.py
"""Own primitives (no Codex code): edge-marker projection with combination."""
import sys; import os;sys.path.insert(0,os.path.dirname(os.path.abspath(__file__)))
import gdsl
from gdsl import H, W, bg_of

EDGES=('top','bottom','left','right')
def edge_marks(g,bg,e):
    h,w=H(g),W(g)
    if e=='top': return {c for c in range(w) if g[0][c]!=bg}
    if e=='bottom': return {c for c in range(w) if g[h-1][c]!=bg}
    if e=='left': return {r for r in range(h) if g[r][0]!=bg}
    return {r for r in range(h) if g[r][w-1]!=bg}
def edge_colour(g,bg,e,i):
    h,w=H(g),W(g)
    return {'top':g[0][i] if e=='top' else None,'bottom':g[h-1][i] if e=='bottom' else None,
            'left':g[i][0] if e=='left' else None,'right':g[i][w-1] if e=='right' else None}[e]

def added_colour(train):
    """The single colour painted onto background cells in every pair (role: new paint)."""
    cs=None
    for p in train:
        i,o=p['input'],p['output']
        if (H(i),W(i))!=(H(o),W(o)): return None
        bg=bg_of(i)
        ch={o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x]!=o[y][x] and i[y][x]==bg}
        cs=ch if cs is None else cs|ch
    return next(iter(cs)) if cs and len(cs)==1 else None

def project(g,rows_from,cols_from,op,paint):
    bg=bg_of(g);h,w=H(g),W(g)
    R=set().union(*[edge_marks(g,bg,e) for e in rows_from]) if rows_from else set()
    C=set().union(*[edge_marks(g,bg,e) for e in cols_from]) if cols_from else set()
    if not R and not C: return None
    used_r={0} if 'top' in cols_from else set(); used_r|={h-1} if 'bottom' in cols_from else set()
    used_c={0} if 'left' in rows_from else set(); used_c|={w-1} if 'right' in rows_from else set()
    out=[r[:] for r in g]
    for y in range(h):
        if y in used_r: continue
        for x in range(w):
            if x in used_c or g[y][x]!=bg: continue
            a,b=y in R,x in C
            hit={'and':a and b,'or':a or b,'xor':a!=b}[op]
            if hit: out[y][x]=paint
    return out

def fam_edge_projection(train):
    i0,o0=train[0]['input'],train[0]['output']
    if (H(i0),W(i0))!=(H(o0),W(o0)): return
    paint=added_colour(train)
    if paint is None: return
    # which edges carry markers in every training input (role: marker edges)
    def edges_with(g): bg=bg_of(g); return {e for e in EDGES if edge_marks(g,bg,e) and len(edge_marks(g,bg,e))<(W(g) if e in('top','bottom') else H(g))}
    common=None
    for p in train:
        es=edges_with(p['input']); common=es if common is None else common&es
    if not common: return
    rowE=[e for e in ('left','right') if e in common]; colE=[e for e in ('top','bottom') if e in common]
    for op in ('and','or','xor'):
        if rowE and colE:
            yield (f"project:{op}[rows={'+'.join(rowE)},cols={'+'.join(colE)}]",4,lambda g,rowE=rowE,colE=colE,op=op:project(g,rowE,colE,op,paint))
    if rowE: yield (f"project:rows[{'+'.join(rowE)}]",4,lambda g,rowE=rowE:project(g,rowE,[], 'or',paint))
    if colE: yield (f"project:cols[{'+'.join(colE)}]",4,lambda g,colE=colE:project(g,[],colE,'or',paint))

FAMILIES=(fam_edge_projection,)


In [ ]:
%%writefile /kaggle/working/probe/fam_move_by_vector.py
"""move.by_vector: every mover object is translated rigidly by a vector computed from its own properties.

One parametrised primitive  translate(seg, sel, rule, edge, collide):
  seg     : object segmentation   c4 | c8 (single colour) | m8 (multicolour 8-conn)
  sel     : which objects move    all | nonstatic-colour | interior (not touching grid edge) | shared-colour
  rule    : v(obj) = unit(obj) * mag(obj)
              unit : one of 8 constant directions | toward nearest anchor | toward same-colour anchor
                     | toward grid centre | toward attached marker
              mag  : 1 | 2 | 3 | cell count | width | height | number of marker cells
            or v(obj) = table[key(obj)], key in {const, colour, part-role in composite (top/mid/bottom)}
  edge    : clip (cells leaving the grid vanish) | wrap (toroidal)
  collide : over (movers paint over) | block (advance at most |v| unit steps while landing on background)
Markers = colours present in inputs but absent from outputs (induced); they are erased, or absorbed into the
object whose bounding box contains them (recoloured) when the rule is 'toward attached marker'.
All parameters are induced from training pairs; candidates are pre-filtered with per-object displacement sets
and fully verified on all training pairs before being yielded.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox, static_colours

DIR8 = {"N": (-1, 0), "S": (1, 0), "W": (0, -1), "E": (0, 1), "NW": (-1, -1), "NE": (-1, 1), "SW": (1, -1), "SE": (1, 1)}
SEGS = ("c4", "c8", "m8")
SELS = ("all", "nonstatic", "interior", "shared")
UNITS = tuple(DIR8) + ("anchor", "anchor_same", "centre", "marker")
MAGS = (1, 2, 3, "size", "w", "h", "mk")
TABLES = ("const", "colour", "role")


def sgn(v): return (v > 0) - (v < 0)


def marker_colours(train):
    m = None
    for p in train:
        ci = {v for r in p["input"] for v in r}; co = {v for r in p["output"] for v in r}
        m = (ci - co) if m is None else (m & (ci - co)) | (m - ci)
    return m or set()


def segment(g, bg, seg, markers):
    gg = [[bg if v in markers else v for v in r] for r in g]
    obs = objects(gg, bg, diag=seg != "c4", by_colour=seg != "m8")
    return [[(y, x, g[y][x]) for y, x in ob] for ob in obs]


def select(obs, g, sel, st):
    h, w = H(g), W(g)
    if sel == "all": mv = [True] * len(obs)
    elif sel == "nonstatic": mv = [not all(c in st for _, _, c in ob) for ob in obs]
    elif sel == "interior": mv = [not any(y in (0, h - 1) or x in (0, w - 1) for y, x, _ in ob) for ob in obs]
    else:
        cnt = Counter(frozenset(c for _, _, c in ob) for ob in obs)
        mv = [cnt[frozenset(c for _, _, c in ob)] >= 2 for ob in obs]
    return [ob for ob, m in zip(obs, mv) if m], [ob for ob, m in zip(obs, mv) if not m]


def attach_markers(g, movers, markers):
    """Marker cells inside a mover's bounding box join it, recoloured to the mover's main colour."""
    h, w = H(g), W(g); out = []
    for ob in movers:
        cells = {(y, x) for y, x, _ in ob}
        main = Counter(c for _, _, c in ob).most_common(1)[0][0]
        r0, c0, r1, c1 = bbox(list(cells))
        mk = {(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if g[y][x] in markers}
        out.append((ob + [(y, x, main) for y, x in mk], mk))
    return out


def centre(cells):
    r0, c0, r1, c1 = bbox([(y, x) for y, x, *_ in cells])
    return (r0 + r1) / 2, (c0 + c1) / 2, (r0, c0, r1, c1)


def toward(ob, target_cells):
    """Unit step from object toward target; the component along an axis where projections overlap is 0."""
    _, _, (r0, c0, r1, c1) = centre(ob)
    _, _, (a0, b0, a1, b1) = centre(target_cells)
    dy = 0 if (a0 <= r1 and r0 <= a1) else sgn(a0 - r0)
    dx = 0 if (b0 <= c1 and c0 <= b1) else sgn(b0 - c0)
    return dy, dx


class Ctx:
    """Everything about one grid needed to evaluate a rule."""
    def __init__(self, g, seg, sel, markers, st, attach):
        self.g = g; self.bg = bg = bg_of(g); self.markers = markers
        obs = segment(g, bg, seg, markers)
        mov, rest = select(obs, g, sel, st)
        if attach:
            am = attach_markers(g, mov, markers)
            mov = [a for a, _ in am]; self.mk = [m for _, m in am]
        else:
            self.mk = [set() for _ in mov]
        # role items: split multicolour movers into colour parts
        self.parts = None
        if seg == "m8":
            parts = []
            for ob in mov:
                byc = {}
                for y, x, c in ob: byc.setdefault(c, []).append((y, x, c))
                ps = sorted(byc.values(), key=lambda p: centre(p)[0])
                for k, p in enumerate(ps):
                    rl = "solo" if len(ps) == 1 else "top" if k == 0 else "bottom" if k == len(ps) - 1 else "mid"
                    parts.append((p, rl))
            self.parts = parts
        self.movers, self.anchors = mov, rest
        self.nmk = sum(v in markers for r in g for v in r)

    def unit(self, i, u):
        ob = self.movers[i]
        if u in DIR8: return DIR8[u]
        if u == "centre":
            h, w = H(self.g), W(self.g)
            return toward(ob, [((h - 1) // 2, (w - 1) // 2), (h // 2, w // 2)])
        if u == "marker":
            if not self.mk[i]: return (0, 0)
            cy, cx, _ = centre(ob); my, mx, _ = centre(list(self.mk[i]))
            return sgn(my - cy), sgn(mx - cx)
        cands = self.anchors
        if u == "anchor_same":
            cs = {c for _, _, c in ob}
            cands = [a for a in cands if {c for _, _, c in a} & cs]
        if not cands: return (0, 0)
        cy, cx, _ = centre(ob)
        best = min(cands, key=lambda a: min(abs(y - cy) + abs(x - cx) for y, x, _ in a))
        return toward(ob, best)

    def mag(self, i, m):
        if isinstance(m, int): return m
        ob = self.movers[i]; r0, c0, r1, c1 = centre(ob)[2]
        return {"size": len(ob), "w": c1 - c0 + 1, "h": r1 - r0 + 1, "mk": self.nmk}[m]

    def items(self, table):
        """(cells, key) list for table rules."""
        if table == "role":
            return [(p, r) for p, r in self.parts] if self.parts is not None else None
        if table == "const": return [(ob, 0) for ob in self.movers]
        return [(ob, ob[0][2]) for ob in self.movers if len({c for _, _, c in ob}) == 1] \
            if all(len({c for _, _, c in ob}) == 1 for ob in self.movers) else None


def paint(g, bg, items, edge, collide):
    """items: list of (cells, (dy,dx), unit or None, steps)."""
    h, w = H(g), W(g); out = [r[:] for r in g]
    for cells, *_ in items:
        for y, x, _ in cells: out[y][x] = bg
    base = [r[:] for r in out]
    for cells, (dy, dx), unit, steps in items:
        if collide == "block":
            k = 0
            while k < steps:
                ny, nx = (k + 1) * unit[0], (k + 1) * unit[1]
                ok = True
                for y, x, _ in cells:
                    yy, xx = y + ny, x + nx
                    if not (0 <= yy < h and 0 <= xx < w) or base[yy][xx] != bg: ok = False; break
                if not ok: break
                k += 1
            dy, dx = k * unit[0], k * unit[1]
        for y, x, c in cells:
            yy, xx = y + dy, x + dx
            if edge == "wrap": yy %= h; xx %= w
            if 0 <= yy < h and 0 <= xx < w: out[yy][xx] = c
    return out


def disp_ok(o, cells, d):
    h, w = H(o), W(o); dy, dx = d
    for y, x, c in cells:
        yy, xx = y + dy, x + dx
        if 0 <= yy < h and 0 <= xx < w and o[yy][xx] != c: return False
    return True


def fam_move_by_vector(train):
    i0, o0 = train[0]["input"], train[0]["output"]
    if any((H(p["input"]), W(p["input"])) != (H(p["output"]), W(p["output"])) for p in train): return
    if all(p["input"] == p["output"] for p in train): return
    markers = marker_colours(train)
    st = static_colours(train)
    yielded = 0
    for seg in SEGS:
        for sel in SELS:
            if sel == "nonstatic" and not st: continue
            for attach in ((False, True) if markers else (False,)):
                ctxs = []
                for p in train:
                    c = Ctx(p["input"], seg, sel, markers, st, attach)
                    if not c.movers or len(c.movers) > 40: ctxs = None; break
                    ctxs.append(c)
                if not ctxs: continue
                progs = []
                # ---- unit * magnitude rules
                units = [u for u in UNITS if (u != "marker") == (not attach)]
                if attach: units = ["marker"]
                for u in units:
                    if u == "anchor_same" and seg == "m8": continue
                    ulist = [[c.unit(k, u) for k in range(len(c.movers))] for c in ctxs]
                    if all(v == (0, 0) for ul in ulist for v in ul): continue
                    for m in MAGS:
                        if m == "mk" and (not markers or attach): continue
                        mlist = [[c.mag(k, m) for k in range(len(c.movers))] for c in ctxs]
                        # prefilter: exact displacement must be consistent (ignoring blocking)
                        pre = all(disp_ok(p["output"], c.movers[k], (ul[k][0] * ml[k], ul[k][1] * ml[k]))
                                  for p, c, ul, ml in zip(train, ctxs, ulist, mlist) for k in range(len(c.movers)))
                        modes = [("over", "clip"), ("over", "wrap")] if pre else []
                        if not isinstance(m, int) or m <= 3: modes.append(("block", "clip"))
                        for collide, edge in modes:
                            progs.append((f"{u}*{m}", u, m, None, None, collide, edge))
                # ---- table rules: vector looked up by key
                for tb in TABLES:
                    if tb == "role" and seg != "m8": continue
                    if attach: continue
                    tabs = {}; good = True
                    for p, c in zip(train, ctxs):
                        its = c.items(tb)
                        if its is None: good = False; break
                        for cells, key in its:
                            h, w = H(p["output"]), W(p["output"])
                            y0, x0, col = cells[0]
                            D = {(yy - y0, xx - x0) for yy in range(h) for xx in range(w) if p["output"][yy][xx] == col}
                            D = {d for d in D if disp_ok(p["output"], cells, d)}
                            tabs[key] = tabs[key] & D if key in tabs else D
                            if not tabs[key]: good = False; break
                        if not good: break
                    if not good or not tabs: continue
                    tab = {k: min(v, key=lambda d: (abs(d[0]) + abs(d[1]), d)) for k, v in tabs.items()}
                    if all(v == (0, 0) for v in tab.values()): continue
                    if tb == "const" and sum(v != (0, 0) for v in tab.values()) == 0: continue
                    for edge in ("clip", "wrap"):
                        progs.append((f"{tb}", None, None, tb, tab, "over", edge))
                for name, u, m, tb, tab, collide, edge in progs:
                    fn = make(seg, sel, attach, markers, st, u, m, tb, tab, collide, edge)
                    if all(fn(p["input"]) == p["output"] for p in train):
                        yield (f"move-by-vector:{name}[{seg},{sel},{collide},{edge}{',attach' if attach else ''}]",
                               (4 if tb is None else 5) + (edge == 'wrap'), fn)
                        yielded += 1
                        if yielded >= 6: return
                        break


def make(seg, sel, attach, markers, st, u, m, tb, tab, collide, edge):
    def fn(g):
        c = Ctx(g, seg, sel, markers, st, attach)
        if not c.movers: return None
        g2 = [[c.bg if v in markers else v for v in r] for r in g]
        if tb is None:
            items = []
            for k, ob in enumerate(c.movers):
                un = c.unit(k, u); mg = c.mag(k, m)
                items.append((ob, (un[0] * mg, un[1] * mg), un, mg))
        else:
            its = c.items(tb)
            if its is None: return None
            items = []
            for cells, key in its:
                if key not in tab: return None
                items.append((cells, tab[key], None, 0))
        return paint(g2, c.bg, items, edge, collide)
    return fn


FAMILIES = (fam_move_by_vector,)


In [ ]:
%%writefile /kaggle/working/probe/fam_move_slide_until_contact.py
"""Own primitive (no Codex code): slide-until-contact.

One parametrised primitive: every mover unit is translated along one of 4 directions until the next
step would hit a blocking cell (contact mode) or leave the grid, or (through mode) straight to the
grid edge ignoring obstacles, optionally painting the swept cells with an induced trail colour.
All parameters are induced from the training pairs:
  movers   : a colour set (a single colour whose cells change, or the whole set of changing colours)
  objects  : 4-conn single colour | 8-conn single colour | 4-conn multi colour
  unit     : whole object | 1-wide strips parallel to the motion axis
  floor    : global background | local background (the colour the unit sits on: region-aware)
  direction: const d | map(key->d) with key in {colour, hollow, open-concavity, size-threshold,
             region colour} | context {away from the anchor it touches, toward the full static line,
             away from the grid edge it touches}; 'stay' is an allowed map value
  recolour : optional map(key->colour) when moved units change colour by class
  stop     : contact | through(+trail colour)
The map / context is fitted from observed per-unit displacements in the training outputs; the search
harness then verifies the resulting program exactly on all pairs.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import product
from gdsl import H, W, bg_of, fit_cmap, apply_cmap, static_colours

DIRS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
OPP = {'up': 'down', 'down': 'up', 'left': 'right', 'right': 'left'}
N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))


# ------------------------------------------------------------------ objects and units
def components(g, M, diag, byc):
    h, w = H(g), W(g); seen = set(); out = []
    nb = N4 + (((-1, -1), (-1, 1), (1, -1), (1, 1)) if diag else ())
    for r in range(h):
        for c in range(w):
            if (r, c) in seen or g[r][c] not in M: continue
            col = g[r][c]; st = [(r, c)]; seen.add((r, c)); cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in seen and g[yy][xx] in M \
                            and (not byc or g[yy][xx] == col):
                        seen.add((yy, xx)); st.append((yy, xx))
            out.append(sorted(cells))
    return out


def strips(cells, axis):
    """Split an object into maximal contiguous 1-wide runs along axis ('v' = columns, 'h' = rows)."""
    s = set(cells); out = []
    for y, x in sorted(cells, key=(lambda p: (p[1], p[0])) if axis == 'v' else (lambda p: p)):
        prev = (y - 1, x) if axis == 'v' else (y, x - 1)
        if prev in s: continue
        run = []; cy, cx = y, x
        while (cy, cx) in s:
            run.append((cy, cx)); cy, cx = (cy + 1, cx) if axis == 'v' else (cy, cx + 1)
        out.append(run)
    return out


# ------------------------------------------------------------------ unit features
def floor_of(g, cells, allmov, gbg, local):
    if not local: return gbg
    s = set(cells); cnt = Counter()
    for y, x in cells:
        for dy, dx in N4:
            yy, xx = y + dy, x + dx
            if 0 <= yy < H(g) and 0 <= xx < W(g) and (yy, xx) not in s and (yy, xx) not in allmov:
                cnt[g[yy][xx]] += 1
    return cnt.most_common(1)[0][0] if cnt else gbg


def bbox_bg_split(cells):
    """(has enclosed hole, has open concavity) wrt the object's bounding box."""
    s = set(cells); ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    r0, r1, c0, c1 = min(ys), max(ys), min(xs), max(xs)
    empty = {(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if (y, x) not in s}
    if not empty: return False, False
    outside = {(y, x) for (y, x) in empty if y in (r0, r1) or x in (c0, c1)}
    st = list(outside); seen = set(outside)
    while st:
        y, x = st.pop()
        for dy, dx in N4:
            p = (y + dy, x + dx)
            if p in empty and p not in seen: seen.add(p); st.append(p)
    return len(seen) < len(empty), bool(seen)


def key_fns():
    return {
        'colour': lambda u: tuple(sorted({u['g'][y][x] for y, x in u['cells']})),
        'hollow': lambda u: u['shape'][0],
        'open': lambda u: u['shape'][1],
        'region': lambda u: u['floor'],
    }


# ------------------------------------------------------------------ contextual directions
def ctx_away_contact(u):
    g, s, fl = u['g'], set(u['cells']), u['floor']; sides = set()
    for y, x in u['cells']:
        for d, (dy, dx) in DIRS.items():
            yy, xx = y + dy, x + dx
            if 0 <= yy < H(g) and 0 <= xx < W(g) and (yy, xx) not in s and (yy, xx) not in u['allmov'] \
                    and g[yy][xx] != fl:
                sides.add(d)
    return OPP[next(iter(sides))] if len(sides) == 1 else 'stay'


def full_lines(g, fl):
    rows = [r for r in range(H(g)) if len(set(g[r])) == 1 and g[r][0] != fl]
    cols = [c for c in range(W(g)) if len({g[r][c] for r in range(H(g))}) == 1 and g[0][c] != fl]
    return rows, cols


def ctx_toward_line(u):
    rows, cols = u['lines']; ys = [y for y, _ in u['cells']]; xs = [x for _, x in u['cells']]
    best = None
    for r in rows:
        if r > max(ys): cand = (r - max(ys), 'down')
        elif r < min(ys): cand = (min(ys) - r, 'up')
        else: continue
        if best is None or cand[0] < best[0]: best = cand
    for c in cols:
        if c > max(xs): cand = (c - max(xs), 'right')
        elif c < min(xs): cand = (min(xs) - c, 'left')
        else: continue
        if best is None or cand[0] < best[0]: best = cand
    return best[1] if best else 'stay'


def ctx_away_edge(u):
    g = u['g']; ys = [y for y, _ in u['cells']]; xs = [x for _, x in u['cells']]
    t = [d for d, hit in (('down', min(ys) == 0), ('up', max(ys) == H(g) - 1),
                           ('right', min(xs) == 0), ('left', max(xs) == W(g) - 1)) if hit]
    return t[0] if len(t) == 1 else 'stay'


CTX = {'away-contact': ctx_away_contact, 'toward-line': ctx_toward_line, 'away-edge': ctx_away_edge}


# ------------------------------------------------------------------ units of a grid under a config
def make_units(g, M, omode, local, unit):
    gbg = bg_of(g)
    kind, arg = M
    M = {v for r in g for v in r} - {gbg}
    if kind == 'excl': M -= arg
    elif kind == 'only': M &= arg
    if gbg in M: return None
    diag, byc = omode
    objs = components(g, M, diag, byc)
    if not objs or len(objs) > 40: return None
    allmov = {p for ob in objs for p in ob}
    units = []
    for ob in objs:
        parts = [ob] if unit == 'obj' else strips(ob, unit)
        for cells in parts:
            fl = floor_of(g, cells, allmov, gbg, local)
            units.append({'g': g, 'cells': cells, 'floor': fl, 'allmov': allmov,
                          'size': len(ob), 'shape': bbox_bg_split(ob)})
    lines = full_lines(g, gbg)
    incol = {v for r in g for v in r}
    for u in units: u['lines'] = lines; u['incol'] = incol
    return units


# ------------------------------------------------------------------ simulation
def simulate(g, units, dirs, stop, trail, recol):
    h, w = H(g), W(g)
    out = [r[:] for r in g]
    pos = []
    for u, d in zip(units, dirs):
        pos.append([(y, x, g[y][x]) for y, x in u['cells']])
    if stop == 'through':
        paint = []
        for u, d, cur in zip(units, dirs, pos):
            for y, x, _ in cur: out[y][x] = u['floor']
        for i, (u, d) in enumerate(zip(units, dirs)):
            if d == 'stay': paint.append(pos[i]); continue
            dy, dx = DIRS[d]; cur = pos[i]; swept = []
            while all(0 <= y + dy < h and 0 <= x + dx < w for y, x, _ in cur):
                swept += [(y, x) for y, x, _ in cur]
                cur = [(y + dy, x + dx, c) for y, x, c in cur]
            if trail is not None:
                for y, x in swept: out[y][x] = trail
            paint.append(cur)
        for i, cur in enumerate(paint):
            for y, x, c in cur: out[y][x] = recol[i] if recol and recol[i] is not None else c
        return out
    # contact: step-wise simultaneous-ish settling, leading units first
    order = sorted(range(len(units)), key=lambda i: 0 if dirs[i] == 'stay' else
                   -max(y * DIRS[dirs[i]][0] + x * DIRS[dirs[i]][1] for y, x, _ in pos[i]))
    moving = True; guard = 0
    while moving and guard < 64:
        moving = False; guard += 1
        for i in order:
            d = dirs[i]
            if d == 'stay': continue
            dy, dx = DIRS[d]; cur = pos[i]; own = {(y, x) for y, x, _ in cur}; fl = units[i]['floor']
            k = 0
            while True:
                nxt = [(y + dy, x + dx) for y, x, _ in cur]
                if any(not (0 <= y < h and 0 <= x < w) or ((y, x) not in own and out[y][x] != fl)
                       for y, x in nxt):
                    break
                for y, x, _ in cur: out[y][x] = fl
                cur = [(y + dy, x + dx, c) for y, x, c in cur]
                for y, x, c in cur: out[y][x] = c
                own = {(y, x) for y, x, _ in cur}; k += 1
            if k: moving = True; pos[i] = cur
    if recol:
        for i, cur in enumerate(pos):
            if recol[i] is not None:
                for y, x, _ in cur: out[y][x] = recol[i]
    return out


# ------------------------------------------------------------------ induction from training
def observe(u, o):
    """Directions (and resulting uniform colour, or None if colours kept) consistent with the output."""
    g = u['g']; h, w = H(g), W(g); res = {}
    cells = u['cells']
    if all(o[y][x] == g[y][x] for y, x in cells): res['stay'] = None
    for d, (dy, dx) in DIRS.items():
        for k in range(1, max(h, w)):
            sh = [(y + dy * k, x + dx * k) for y, x in cells]
            if any(not (0 <= y < h and 0 <= x < w) for y, x in sh): break
            if all(o[y2][x2] == g[y][x] for (y, x), (y2, x2) in zip(cells, sh)):
                res.setdefault(d, None)
            else:
                cs = {o[y2][x2] for y2, x2 in sh}
                if len(cs) == 1 and len({g[y][x] for y, x in cells}) == 1:
                    c = next(iter(cs))
                    if c not in u['incol'] and d not in res: res[d] = c
    return res


def mover_sets(train):
    """Mover colour roles: all non-background | non-background minus the static colours | one colour
    whose cells change."""
    ch = set()
    for p in train:
        i, o = p['input'], p['output']
        for y in range(H(i)):
            for x in range(W(i)):
                if i[y][x] != o[y][x]: ch.add(i[y][x])
    out = [('nonbg', None)]
    st = frozenset(static_colours(train) - {bg_of(p['input']) for p in train})
    if st: out.append(('excl', st))
    out += [('only', frozenset([c])) for c in sorted(ch)]
    return out


def allowed(ob):
    """Directions consistent with an observation: a unit seen in place may have been blocked."""
    return set(ob) | (set(DIRS) if 'stay' in ob else set())


def colour_for(ob, d):
    return ob[d] if d in ob else ob.get('stay')


def fit(train_units, obs, keyname):
    """Yield (dir source, recolour map or None) consistent with the observations."""
    flat = [(u, ob) for us, os_ in zip(train_units, obs) for u, ob in zip(us, os_)]
    if keyname in CTX:
        f = CTX[keyname]; rec = {}
        for u, ob in flat:
            d = f(u)
            if d not in allowed(ob): return
            c = colour_for(ob, d)
            if rec.setdefault(d, c) != c: return
        yield 'ctx', (rec if any(v is not None for v in rec.values()) else None)
        return
    if keyname.startswith('size>'):
        t = int(keyname[5:]); kf = lambda u: u['size'] > t
    elif keyname == 'const':
        kf = lambda u: 0
    else:
        kf = key_fns()[keyname]
    cand = {}
    for u, ob in flat:
        k = kf(u); a = allowed(ob)
        cand[k] = cand[k] & a if k in cand else a
        if not cand[k]: return
    opts = {}
    for k, s in cand.items():
        mv = sorted(s - {'stay'})
        opts[k] = ['stay'] if 'stay' in s and len(mv) == 4 else mv
        if not opts[k]: return
    ks = sorted(opts, key=str); n = 0
    for combo in product(*[opts[k] for k in ks]):
        dmap = dict(zip(ks, combo))
        if all(v == 'stay' for v in dmap.values()): continue
        rec = {}; ok = True
        for u, ob in flat:
            k = kf(u); c = colour_for(ob, dmap[k])
            if rec.setdefault(k, c) != c: ok = False; break
        if not ok: continue
        yield (kf, dmap), (rec if any(v is not None for v in rec.values()) else None)
        n += 1
        if n >= 4: return


def fam_slide_until_contact(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    if all(p['input'] == p['output'] for p in train): return
    yielded = 0
    for M in mover_sets(train):
        for omode in ((False, True), (True, True), (False, False)):
            for local in (False, True):
                for unit in ('obj', 'v', 'h'):
                    TU = [make_units(p['input'], M, omode, local, unit) for p in train]
                    if any(t is None for t in TU): continue
                    OB = [[observe(u, p['output']) for u in us] for us, p in zip(TU, train)]
                    if any(not ob for os_ in OB for ob in os_): continue
                    sizes = sorted({u['size'] for us in TU for u in us})
                    keys = ['const', 'colour', 'hollow', 'open', 'region'] + \
                           ['size>%d' % s for s in sizes[:-1]] + list(CTX)
                    if unit != 'obj':
                        keys = ['const', 'colour', 'region'] + list(CTX)
                    for kn in keys:
                      for dsrc, rec in fit(TU, OB, kn):
                        for stop in ('contact', 'through'):
                            trail = None
                            prog = make_prog(M, omode, local, unit, kn, dsrc, rec, stop, None)
                            if stop == 'through':
                                trail = induce_trail(train, prog)
                                if trail == 'fail': continue
                                prog = make_prog(M, omode, local, unit, kn, dsrc, rec, stop, trail)
                            if not fits(train, prog): continue
                            name = 'slide:%s[movers=%s,%s%s,%s,unit=%s,dir=%s%s%s]' % (
                                stop, M[0] if M[0] == 'nonbg' else M[0] + '|'.join(map(str, sorted(M[1]))), '8' if omode[0] else '4',
                                '' if omode[1] else ',multi', 'local' if local else 'global', unit, kn,
                                '' if dsrc == 'ctx' else ':' + ','.join('%s>%s' % kv for kv in sorted(dsrc[1].items(), key=str)),
                                ',recolour' if rec else '') + ('' if trail is None else '+trail%d' % trail)
                            yield (name, 4 + (kn != 'const') + (rec is not None) + (trail is not None), prog)
                            yielded += 1
                            if yielded >= 8: return


def make_prog(M, omode, local, unit, kn, dsrc, rec, stop, trail):
    def fn(g):
        units = make_units(g, M, omode, local, unit)
        if not units: return None
        if dsrc == 'ctx':
            dirs = [CTX[kn](u) for u in units]
        else:
            kf, dmap = dsrc
            ks = [kf(u) for u in units]
            if any(k not in dmap for k in ks): return None
            dirs = [dmap[k] for k in ks]
        if all(d == 'stay' for d in dirs): return None
        recol = None
        if rec:
            if dsrc == 'ctx': recol = [rec.get(d) for d in dirs]
            else:
                if any(k not in rec for k in ks): return None
                recol = [rec[k] for k in ks]
        return simulate(g, units, dirs, stop, trail, recol)
    return fn


def fits(train, prog):
    preds = []
    for p in train:
        try: r = prog(p['input'])
        except Exception: return False
        if r is None: return False
        preds.append(r)
    outs = [p['output'] for p in train]
    if preds == outs: return True
    m = fit_cmap(preds, outs)
    return bool(m) and all(apply_cmap(a, m) == b for a, b in zip(preds, outs))


def induce_trail(train, prog):
    """Through mode: the one colour painted on swept cells, or None when no trail is needed."""
    cs = set()
    for p in train:
        r = prog(p['input'])
        if r is None: return 'fail'
        o = p['output']
        cs |= {o[y][x] for y in range(H(o)) for x in range(W(o)) if r[y][x] != o[y][x]}
    if not cs: return None
    return next(iter(cs)) if len(cs) == 1 else 'fail'


FAMILIES = (fam_slide_until_contact,)


In [ ]:
%%writefile /kaggle/working/probe/fam_stamp_local_stencil.py
"""STAMP.local_stencil: every seed (cell / object) gets a fixed relative stencil painted around it.

One parametrised primitive. Everything below is induced from the training pairs:
  seg    : how seeds are segmented   cell | obj4 | obj8 | multi4 | multi8 (non-background components)
  frame  : local coordinate frame    tl (bbox top-left) | near (offset from nearest bbox side, size-free)
           | d8 (shape canonicalised up to rotation/reflection; stencil turns with it)
           | wall (frame rotated so that the nearest uniform border line is "up")
  key    : seed signature selecting the stencil: () | colour | shape | colour+shape | square
           | unique-colour | colour+inside-other-object
  stencil: learned map canonical-offset -> colour spec, spec = constant colour | colour of the seed's
           own cell at a canonical position | colour of the nearest border line.  Offsets on the seed's
           own cells express erase / recolour of the seed.
  paint  : all | bg-only | protect (never overwrite a stamping seed)
The learner aggregates, per (key, offset), what the output holds in every training instance and keeps an
offset only when one spec explains >=75% of instances and at least two of them are real changes.
Programs are yielded only when they reproduce every training output exactly.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter, defaultdict
from gdsl import H, W, bg_of, objects, bbox

R = 2          # stencil radius around the seed bbox
MAXREF = 9     # seeds larger than this cannot be used as colour references
D8T = [(sw, fy, fx) for sw in (0, 1) for fy in (0, 1) for fx in (0, 1)]


def fwd(t, dy, dx, h, w):
    sw, fy, fx = t
    a = h - 1 - dy if fy else dy
    b = w - 1 - dx if fx else dx
    return (b, a) if sw else (a, b)


def dims(t, h, w):
    return (w, h) if t[0] else (h, w)


def anchor(a, n, near):
    if not near: return a
    return ('t', a) if 2 * a <= n - 1 else ('b', a - (n - 1))


def walls(g, bg):
    h, w = H(g), W(g); out = {}
    for side, line in (('top', g[0]), ('bottom', g[h - 1]), ('left', [r[0] for r in g]), ('right', [r[w - 1] for r in g])):
        if len(set(line)) == 1 and line[0] != bg: out[side] = line[0]
    return out


WALL_T = {'top': (0, 0, 0), 'bottom': (0, 1, 0), 'left': (1, 0, 0), 'right': (1, 0, 1)}


def segment(g, bg, seg):
    if seg == 'cell':
        return [[(y, x)] for y in range(H(g)) for x in range(W(g)) if g[y][x] != bg]
    diag = seg.endswith('8'); byc = seg.startswith('obj')
    return objects(g, bg, diag, byc)


def seed_infos(g, seg, frame):
    """Per seed: cells, bbox, transform, canonical dims, features, wall colour.  None if the frame is undefined."""
    bg = bg_of(g); h, w = H(g), W(g)
    seeds = segment(g, bg, seg)
    if not seeds or len(seeds) > 400: return None, bg
    wl = walls(g, bg) if frame == 'wall' else None
    if frame == 'wall' and not wl: return None, bg
    colsets = [tuple(sorted({g[y][x] for y, x in s})) for s in seeds]
    ccount = Counter(colsets)
    big = None
    infos = []
    for s, cs in zip(seeds, colsets):
        r0, c0, r1, c1 = bbox(s); sh, sw_ = r1 - r0 + 1, c1 - c0 + 1
        rel = [(y - r0, x - c0) for y, x in s]
        wcol = None
        if frame == 'd8':
            t = min(D8T, key=lambda t: sorted(fwd(t, a, b, sh, sw_) for a, b in rel))
        elif frame == 'wall':
            d = {'top': r0, 'bottom': h - 1 - r1, 'left': c0, 'right': w - 1 - c1}
            ds = sorted((d[k], k) for k in wl)
            if len(ds) > 1 and ds[0][0] == ds[1][0]: t = None
            else: t = WALL_T[ds[0][1]]; wcol = wl[ds[0][1]]
        else:
            t = (0, 0, 0)
        if t is None:
            infos.append(None); continue
        ch, cw = dims(t, sh, sw_); near = frame == 'near'
        canon = {}
        for (a, b), (y, x) in zip(rel, s):
            p = fwd(t, a, b, sh, sw_)
            canon[(anchor(p[0], ch, near), anchor(p[1], cw, near))] = g[y][x]
        shape = tuple(sorted(fwd(t, a, b, sh, sw_) for a, b in rel))
        feat = {'colour': cs, 'shape': shape, 'square': sh == sw_ and sh >= 2,
                'unique': ccount[cs] == 1}
        infos.append(dict(cells=s, box=(r0, c0, r1, c1), t=t, cd=(ch, cw), canon=canon, feat=feat, wcol=wcol,
                          cs=cs))
    # inside the bbox of a (>=4 cell) object of another colour
    if any(i is not None for i in infos):
        if big is None:
            big = [(bbox(o), g[o[0][0]][o[0][1]]) for o in objects(g, bg, False, True) if len(o) >= 4]
        for inf in infos:
            if inf is None: continue
            r0, c0, r1, c1 = inf['box']
            inf['feat']['inbox'] = any(b[0] <= r0 and r1 <= b[2] and b[1] <= c0 and c1 <= b[3] and col not in inf['cs']
                                       for b, col in big)
    return infos, bg


def window(inf, h, w, near):
    r0, c0, r1, c1 = inf['box']; sh, sw_ = r1 - r0 + 1, c1 - c0 + 1
    t = inf['t']; ch, cw = inf['cd']
    for y in range(max(0, r0 - R), min(h, r1 + R + 1)):
        for x in range(max(0, c0 - R), min(w, c1 + R + 1)):
            a, b = fwd(t, y - r0, x - c0, sh, sw_)
            yield y, x, (anchor(a, ch, near), anchor(b, cw, near))


def keyof(inf, key):
    return tuple(inf['feat'][k] for k in key)


KEYS = [(), ('colour',), ('shape',), ('colour', 'shape'), ('square',), ('unique',), ('colour', 'inbox')]


def learn(train, seg, frame, keys):
    near = frame == 'near'
    # stats[key][keyval][off] = [n, Counter(spec->match), Counter(spec->change-match)]
    stats = {k: defaultdict(lambda: defaultdict(lambda: [0, Counter(), Counter()])) for k in keys}
    for p in train:
        g, o = p['input'], p['output']
        infos, bg = seed_infos(g, seg, frame)
        if infos is None: return None
        h, w = H(g), W(g)
        for inf in infos:
            if inf is None: continue
            refs = inf['canon'] if len(inf['cells']) <= MAXREF else {}
            kv = [(k, keyof(inf, k)) for k in keys]
            for y, x, off in window(inf, h, w, near):
                ov = o[y][x]; chg = ov != g[y][x]
                specs = [('c', ov)] + [('s', q) for q, v in refs.items() if v == ov]
                if inf['wcol'] is not None and inf['wcol'] == ov: specs.append(('w',))
                for k, v in kv:
                    st = stats[k][v][off]; st[0] += 1
                    for sp in specs:
                        st[1][sp] += 1
                        if chg: st[2][sp] += 1
    res = {}
    for k in keys:
        sten = {}
        for v, offs in stats[k].items():
            m = {}
            for off, (n, mc, cc) in offs.items():
                if not cc: continue
                sp, c = max(mc.items(), key=lambda it: (it[1], cc[it[0]], -len(it[0])))
                if c >= 0.75 * n and cc[sp] >= 2 and 2 * cc[sp] >= c:
                    m[off] = sp
            sten[v] = m
        if any(sten.values()):
            res[k] = sten
    return res


def apply(g, seg, frame, key, sten, mode):
    infos, bg = seed_infos(g, seg, frame)
    if infos is None or any(i is None for i in infos): return None
    h, w = H(g), W(g); near = frame == 'near'
    out = [r[:] for r in g]
    plan = []
    for inf in infos:
        kv = keyof(inf, key)
        if kv not in sten: return None
        m = sten[kv]
        if m: plan.append((inf, m))
    protect = {c for inf, _ in plan for c in inf['cells']}
    ext = []
    for inf, m in plan:
        own = set(inf['cells'])
        for y, x, off in window(inf, h, w, near):
            sp = m.get(off)
            if sp is None: continue
            if sp[0] == 'c': col = sp[1]
            elif sp[0] == 's':
                col = inf['canon'].get(sp[1])
                if col is None: return None
            else: col = inf['wcol']
            if (y, x) in own: out[y][x] = col
            else: ext.append((y, x, col))
    for y, x, col in ext:
        if mode == 'bg' and g[y][x] != bg: continue
        if mode == 'protect' and (y, x) in protect: continue
        out[y][x] = col
    return out


def fam_local_stencil(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    if all(p['input'] == p['output'] for p in train): return
    # every changed cell must lie within Chebyshev R of a non-background input cell
    for p in train:
        g, o = p['input'], p['output']; bg = bg_of(g); h, w = H(g), W(g)
        for y in range(h):
            for x in range(w):
                if g[y][x] != o[y][x] and not any(g[yy][xx] != bg for yy in range(max(0, y - R), min(h, y + R + 1))
                                                  for xx in range(max(0, x - R), min(w, x + R + 1))):
                    return
    outs = [p['output'] for p in train]
    found = 0
    for seg in ('cell', 'obj4', 'obj8', 'multi4', 'multi8'):
        for frame in ('tl', 'wall', 'near', 'd8'):
            if seg == 'cell' and frame in ('near', 'd8'): continue
            if frame == 'd8': keys = [('shape',), ('colour', 'shape')]
            elif seg == 'cell': keys = [k for k in KEYS if 'shape' not in k and 'square' not in k]
            else: keys = KEYS
            if frame == 'near': keys = [k for k in keys if 'shape' not in k]
            try:
                learned = learn(train, seg, frame, keys)
            except Exception:
                learned = None
            if not learned: continue
            for key, sten in learned.items():
                for mode in ('all', 'bg', 'protect'):
                    ok = all(apply(p['input'], seg, frame, key, sten, mode) == p['output'] for p in train)
                    if not ok: continue
                    nk = '+'.join(key) or 'any'
                    cost = 4 + (len(key) > 1) + (frame in ('d8', 'wall', 'near'))
                    yield (f"stencil[{seg},{frame},key={nk},{mode}]", cost,
                           lambda g, seg=seg, frame=frame, key=key, sten=sten, mode=mode: apply(g, seg, frame, key, sten, mode))
                    found += 1
                    break
                if found >= 6: return


FAMILIES = (fam_local_stencil,)


In [ ]:
%%writefile /kaggle/working/probe/fam_summarise_ranked_bars.py
"""summarise.ranked_bars: colour classes are ranked by an induced key and rendered as bars.

One parametrised primitive.  Parameters, all induced from the training pairs by decoding the
outputs (never literal task constants):
  filter  : which colour classes are items   (all | no-grid: drop colours owning a full row/col |
                                               straight: only colours lying in one row or column)
  select  : all | drop-first (after sorting) | keep-max(f) (only items with the largest feature f)
  key/dir : ordering (numeric feature asc/desc, position incl. auto axis, occlusion layering)
  length  : bar length (a numeric feature | 1 | full canvas width)
  mode    : bars (one row per item) | wrap (item starts a row, wraps at width) | pour (continuous)
  canvas  : rows / cols from {#items, max length, fixed size, input H, input W}
  frame   : one of the 8 dihedral maps (bar axis, alignment side, order direction)
Zero-length bars are not drawn (their items vanish), which is how marker colours drop out.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, D8

INV = {"id": "id", "r90": "r270", "r270": "r90", "r180": "r180", "fh": "fh", "fv": "fv", "T": "T", "aT": "aT"}

def _compose_T():
    """name of t∘T for each t (used when the ordering axis is x: canonical grid is transposed first)."""
    probe = [[1, 2, 3], [4, 5, 6]]
    imgs = {k: f(probe) for k, f in D8.items()}
    out = {}
    for k, f in D8.items():
        r = f(D8["T"](probe))
        out[k] = next(n for n, im in imgs.items() if im == r)
    return out
TCOMP = _compose_T()

NUM = ("ncells", "nobj4", "nobj8", "height", "width", "maxobj", "holes", "enclosed")
POS = ("miny", "minx", "cy", "cx", "first", "auto")
KEYS = NUM + ("value",) + POS + ("layer",)
FILTERS = ("all", "no-grid", "straight")

# ------------------------------------------------------------------ item features
def _enclosed(g, c, bg):
    """cells not of colour c that cannot reach the grid border without crossing colour c."""
    h, w = H(g), W(g); seen = [[False] * w for _ in range(h)]; st = []
    for y in range(h):
        for x in range(w):
            if (y in (0, h - 1) or x in (0, w - 1)) and g[y][x] != c:
                seen[y][x] = True; st.append((y, x))
    while st:
        y, x = st.pop()
        for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            yy, xx = y + dy, x + dx
            if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and g[yy][xx] != c:
                seen[yy][xx] = True; st.append((yy, xx))
    hb = sum(1 for y in range(h) for x in range(w) if not seen[y][x] and g[y][x] == bg)
    ho = sum(1 for y in range(h) for x in range(w) if not seen[y][x] and g[y][x] not in (bg, c))
    return hb, ho

def _layers(g, bg, cells):
    """Bottom-to-top occlusion order of colour classes, or None if not uniquely determined."""
    above = {c: set() for c in cells}
    for c, cs in cells.items():
        ys = [y for y, _ in cs]; xs = [x for _, x in cs]
        r0, r1, c0, c1 = min(ys), max(ys), min(xs), max(xs)
        if all(y in (r0, r1) or x in (c0, c1) for y, x in cs):
            exp = [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if y in (r0, r1) or x in (c0, c1)]
        elif r1 - r0 == c1 - c0 and all(y - x == r0 - c0 for y, x in cs):
            exp = [(r0 + k, c0 + k) for k in range(r1 - r0 + 1)]
        elif r1 - r0 == c1 - c0 and all(y + x == r0 + c1 for y, x in cs):
            exp = [(r0 + k, c1 - k) for k in range(r1 - r0 + 1)]
        else:
            exp = [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)]
        for y, x in exp:
            d = g[y][x]
            if d != c and d != bg and d in above: above[c].add(d)
    # Kahn: bottom = colour with nothing below it
    below = {c: {d for d in cells if c in above[d]} for c in cells}
    order, left = [], set(cells)
    while left:
        free = [c for c in left if not (below[c] & left)]
        if len(free) != 1: return None
        order.append(free[0]); left.discard(free[0])
    return order

def features(g, bgspec="mode"):
    bg = bg_of(g) if bgspec == "mode" else -1 if bgspec == "none" else bgspec
    h, w = H(g), W(g)
    cells = {}
    for y in range(h):
        for x in range(w):
            if g[y][x] != bg: cells.setdefault(g[y][x], []).append((y, x))
    o4 = Counter(g[ob[0][0]][ob[0][1]] for ob in objects(g, bg, False, True))
    obs8 = objects(g, bg, True, True)
    o8 = Counter(g[ob[0][0]][ob[0][1]] for ob in obs8)
    mo = {}
    for ob in obs8:
        c = g[ob[0][0]][ob[0][1]]; mo[c] = max(mo.get(c, 0), len(ob))
    F = {}
    for c, cs in cells.items():
        ys = [y for y, _ in cs]; xs = [x for _, x in cs]
        hb, ho = _enclosed(g, c, bg)
        full = any(all(g[y][x] == c for x in range(w)) for y in set(ys)) or any(all(g[y][x] == c for y in range(h)) for x in set(xs))
        cy = sum(ys) / len(ys); cx = sum(xs) / len(xs)
        F[c] = {"ncells": len(cs), "nobj4": o4[c], "nobj8": o8[c], "height": max(ys) - min(ys) + 1,
                "width": max(xs) - min(xs) + 1, "maxobj": mo[c], "holes": hb, "enclosed": ho,
                "value": c, "miny": min(ys), "minx": min(xs), "cy": cy, "cx": cx,
                "first": min(cs), "grid": full, "straight": len(set(ys)) == 1 or len(set(xs)) == 1}
    return {"bg": bg, "F": F, "cells": cells, "h": h, "w": w, "g": g}

def order_items(info, flt, key, desc, sel):
    """Returns (ordered colours, transposed?) or None."""
    F = info["F"]
    items = [c for c in F if flt == "all" or (flt == "no-grid" and not F[c]["grid"]) or (flt == "straight" and F[c]["straight"])]
    if not items: return None
    tr = False
    if key == "layer":
        lay = _layers_cached(info, tuple(sorted(items)))
        if lay is None: return None
        seq = lay[::-1] if desc else lay
    else:
        k = key
        if key == "auto":
            sy = max(F[c]["cy"] for c in items) - min(F[c]["cy"] for c in items)
            sx = max(F[c]["cx"] for c in items) - min(F[c]["cx"] for c in items)
            if sx == sy: return None
            tr = sx > sy; k = "cx" if tr else "cy"
        vals = [F[c][k] for c in items]
        if len(set(vals)) != len(vals): return None      # ambiguous ranking
        seq = sorted(items, key=lambda c: F[c][k], reverse=desc)
    if sel == "drop-first":
        seq = seq[1:]
    elif sel.startswith("max:"):
        f = sel[4:]; m = max(F[c][f] for c in seq); seq = [c for c in seq if F[c][f] == m]
    return seq, tr

def _layers_cached(info, items):
    ck = ("layer", items)
    if ck not in info:
        info[ck] = _layers(info["g"], info["bg"], {c: info["cells"][c] for c in items})
    return info[ck]

# ------------------------------------------------------------------ decode / render
def decode(C, bg, mode):
    """Canonical grid -> list of (colour, length) or None."""
    h, w = H(C), W(C)
    if mode == "pour":
        flat = [v for r in C for v in r]
        while flat and flat[-1] == bg: flat.pop()
        seq = []
        for v in flat:
            if v == bg: return None
            if seq and seq[-1][0] == v: seq[-1][1] += 1
            else: seq.append([v, 1])
        return [tuple(s) for s in seq]
    rows = []
    for r in C:
        L = 0
        while L < w and r[L] != bg: L += 1
        if L and any(v != r[0] for v in r[:L]): return None
        if any(v != bg for v in r[L:]): return None
        rows.append((r[0] if L else None, L))
    while rows and rows[-1][1] == 0: rows.pop()
    if any(L == 0 for _, L in rows): return None
    if mode == "bars": return rows
    seq = []
    for i, (c, L) in enumerate(rows):
        if seq and seq[-1][0] == c and rows[i - 1][1] == w: seq[-1][1] += L
        else: seq.append([c, L])
    return [tuple(s) for s in seq]

def render(seq, R, Cn, bg, mode):
    if R <= 0 or Cn <= 0 or R > 30 or Cn > 30: return None
    out = [[bg] * Cn for _ in range(R)]
    if mode == "bars":
        if len(seq) > R: return None
        for i, (c, L) in enumerate(seq):
            if L > Cn: return None
            for x in range(L): out[i][x] = c
    elif mode == "wrap":
        y = 0
        for c, L in seq:
            while L > 0:
                if y >= R: return None
                k = min(L, Cn)
                for x in range(k): out[y][x] = c
                L -= k; y += 1
    else:
        pos = 0
        for c, L in seq:
            for _ in range(L):
                if pos >= R * Cn: return None
                out[pos // Cn][pos % Cn] = c; pos += 1
    return out

def dim(rule, g, n, lens):
    if rule == "n": return n
    if rule == "maxlen": return max(lens) if lens else 0
    if rule == "inH": return H(g)
    if rule == "inW": return W(g)
    return rule  # fixed int

def program(bgs, flt, key, desc, sel, length, mode, rrule, crule, t):
    def fn(g):
        info = features(g, bgs)
        r = order_items(info, flt, key, desc, sel)
        if r is None: return None
        seq, tr = r
        F = info["F"]
        n = len(seq)
        if length == "full":
            Cn = dim(crule, g, n, [])
            lens = [Cn] * n
        else:
            lens = [1 if length == "one" else F[c][length] for c in seq]
            Cn = dim(crule, g, n, lens)
        items = [(c, L) for c, L in zip(seq, lens) if L > 0]
        if not items: return None
        R = dim(rrule, g, len(items), lens)
        out = render(items, R, Cn, info["bg"], mode)
        if out is None: return None
        tt = TCOMP[t] if tr else t
        return D8[tt](out)
    return fn

# ------------------------------------------------------------------ induction
def _score(sig):
    bgs, mode, t, flt, key, desc, sel, length, rr, cr = sig
    s = (sel != "all") + (flt != "all") + (bgs != "mode") + isinstance(rr, int) + isinstance(cr, int)
    if key == length: pass
    elif key in ("layer", "auto", "value", "first"): s += 0.5
    elif key in POS: s += 1
    else: s += 2
    return s

def fam_ranked_bars(train):
    o0 = train[0]["output"]
    if H(o0) * W(o0) > 400: return
    common = set.intersection(*[{v for r in p["input"] for v in r} for p in train])
    bgspecs = ["mode", "none"] + sorted(c for c in common if any(bg_of(p["input"]) != c for p in train))
    found = []
    for bgs in bgspecs:
        infos = []
        for p in train:
            info = features(p["input"], bgs)
            if len(info["F"]) < 1 or len(info["F"]) > 12: infos = None; break
            infos.append(info)
        if not infos: continue
        dec = {}
        for mode in ("bars", "wrap", "pour"):
            for t in D8:
                ds = []
                for p, info in zip(train, infos):
                    C = D8[INV[t]](p["output"])
                    d = decode(C, info["bg"], mode)
                    if not d: ds = None; break
                    ds.append((d, H(C), W(C)))
                dec[(mode, t)] = ds
        if not any(dec.values()): continue
        sels = ["all", "drop-first"] + ["max:" + f for f in NUM]
        for mode in ("bars", "wrap", "pour"):
            for t in D8:
                if dec[(mode, t)] is None and dec[(mode, TCOMP[t])] is None: continue
                for flt in FILTERS:
                    for key in KEYS:
                        for desc in (False, True):
                            for sel in sels:
                                res = _match(train, infos, dec, mode, t, flt, key, desc, sel)
                                if not res: continue
                                for length, rr, cr in res:
                                    sig = (bgs, mode, t, flt, key, desc, sel, length, rr, cr)
                                    fn = program(bgs, flt, key, desc, sel, length, mode, rr, cr, t)
                                    if all(fn(p["input"]) == p["output"] for p in train):
                                        found.append((_score(sig), len(found), sig, fn))
                                if len(found) > 300: break
    found.sort(key=lambda x: (x[0], x[1]))
    for sc, _, sig, fn in found[:4]:
        bgs, mode, t, flt, key, desc, sel, length, rr, cr = sig
        name = f"ranked-bars:{mode}[bg={bgs},{flt},{sel},key={key}{'-' if desc else '+'},len={length},rows={rr},cols={cr},{t}]"
        yield (name, 5, fn)

def _match(train, infos, dec, mode, t, flt, key, desc, sel):
    per = []
    multi = False
    for p, info in zip(train, infos):
        r = order_items(info, flt, key, desc, sel)
        if r is None: return None
        seq, tr = r
        tt = TCOMP[t] if tr else t
        ds = dec.get((mode, tt))
        if ds is None: return None
        d, R, Cn = ds[len(per)]
        cols = [c for c, _ in d]
        if not set(cols) <= set(seq): return None
        if len(cols) >= 2: multi = True
        per.append((info, seq, d, R, Cn))
    if not multi: return None
    # lengths: items missing from the decoded sequence must have length 0 under the feature
    lens_ok = []
    for length in NUM + ("one", "full"):
        ok = True
        for info, seq, d, R, Cn in per:
            dd = dict(d)
            kept = [c for c in seq if c in dd]
            if [c for c, _ in d] != kept: ok = False; break
            for c in seq:
                if length == "one": L = 1
                elif length == "full": L = Cn
                else: L = info["F"][c][length]
                if dd.get(c, 0) != L: ok = False; break
            if not ok: break
        if ok: lens_ok.append(length)
    if not lens_ok: return None
    out = []
    for length in lens_ok:
        rrs = []; crs = []
        for rule in ("n", "maxlen", "inH", "inW"):
            if all(R == _dimv(rule, info, len(d), [L for _, L in d]) for info, seq, d, R, Cn in per):
                rrs.append(rule)
            if length != "full" and all(Cn == _dimv(rule, info, len(d), [L for _, L in d]) for info, seq, d, R, Cn in per):
                crs.append(rule)
        if len({R for _, _, _, R, _ in per}) == 1: rrs.append(per[0][3])
        if len({Cn for _, _, _, _, Cn in per}) == 1: crs.append(per[0][4])
        if length == "full":
            for rule in ("n", "inH", "inW"):
                if all(Cn == _dimv(rule, info, len(d), []) for info, seq, d, R, Cn in per): crs.append(rule)
        if rrs and crs: out.append((length, rrs[0], crs[0]))
    return out

def _dimv(rule, info, n, lens):
    if rule == "n": return n
    if rule == "maxlen": return max(lens) if lens else -1
    if rule == "inH": return info["h"]
    if rule == "inW": return info["w"]

FAMILIES = (fam_ranked_bars,)


In [ ]:
%%writefile /kaggle/working/probe/gdsl.py
"""G-DSL: whole-grid primitives with parameters inferred from training pairs only.

Every program is verified exactly on all training pairs before it is applied to a test input.
Programs are enumerated in description-length order (cheapest first). No task identifiers,
no task-specific constants: every parameter (colours, factors, selectors, masks) is induced
from the task's own training pairs.
"""
from __future__ import annotations
from collections import Counter
from itertools import product

# ------------------------------------------------------------------ grid helpers
def H(g): return len(g)
def W(g): return len(g[0])
def T(g): return [list(r) for r in zip(*g)]
def fh(g): return [r[::-1] for r in g]
def fv(g): return g[::-1]
def r90(g): return fh(T(g))
def r180(g): return fv(fh(g))
def r270(g): return fv(T(g))
def at(g): return r180(T(g))
D8 = {"id": lambda g: [list(r) for r in g], "r90": r90, "r180": r180, "r270": r270,
      "fh": fh, "fv": fv, "T": T, "aT": at}
def bg_of(g):
    return Counter(v for r in g for v in r).most_common(1)[0][0]
def colours(g): return {v for r in g for v in r}
def eq(a, b): return a == b

def objects(g, bg, diag=False, by_colour=True):
    h, w = H(g), W(g); seen = [[False] * w for _ in range(h)]; out = []
    nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
    for r in range(h):
        for c in range(w):
            if seen[r][c] or g[r][c] == bg: continue
            col = g[r][c]; st = [(r, c)]; seen[r][c] = True; cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and g[yy][xx] != bg and (not by_colour or g[yy][xx] == col):
                        seen[yy][xx] = True; st.append((yy, xx))
            out.append(cells)
    return out

def bbox(cells):
    ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    return min(ys), min(xs), max(ys), max(xs)

def crop(g, b):
    r0, c0, r1, c1 = b
    return [row[c0:c1 + 1] for row in g[r0:r1 + 1]]

# ------------------------------------------------------------------ colour map (post-step)
def fit_cmap(preds, outs):
    """Global colour substitution consistent across all pairs; None if impossible."""
    m = {}
    for p, o in zip(preds, outs):
        if p is None or H(p) != H(o) or W(p) != W(o): return None
        for rp, ro in zip(p, o):
            for a, b in zip(rp, ro):
                if m.setdefault(a, b) != b: return None
    return m

def apply_cmap(g, m):
    """Apply an induced colour substitution; refuse colours never seen in training (no extrapolation)."""
    if g is None: return None
    if any(v not in m for r in g for v in r): return None
    return [[m[v] for v in r] for r in g]

# ------------------------------------------------------------------ primitive families
# Each family yields (name, cost, fn) where fn(grid) -> grid or None. Families may inspect
# training pairs to induce parameters (never test outputs).

def fam_geometric(train):
    for k, f in D8.items():
        if k != "id": yield ("dihedral:" + k, 1, f)

def fam_tile(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) % H(i) or W(o) % W(i): return
    n, m = H(o) // H(i), W(o) // W(i)
    if n * m == 1 or n * m > 16: return
    # per-tile dihedral transform inferred from the first pair, verified later on all pairs
    pat = []
    for a in range(n):
        row = []
        for b in range(m):
            blk = [r[b * W(i):(b + 1) * W(i)] for r in o[a * H(i):(a + 1) * H(i)]]
            ks = [k for k, f in D8.items() if (H(i) == W(i) or k in ("id", "r180", "fh", "fv")) and f(i) == blk]
            if not ks: return
            row.append(ks[0])
        pat.append(row)
    def fn(g, pat=pat, n=n, m=m):
        out = []
        for a in range(n):
            blks = [D8[pat[a][b]](g) for b in range(m)]
            if any(H(x) != H(blks[0]) for x in blks): return None
            for rr in range(H(blks[0])):
                out.append(sum((x[rr] for x in blks), []))
        return out
    yield (f"tile:{n}x{m}", 2, fn)

def fam_scale(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) % H(i) == 0 and W(o) % W(i) == 0 and (H(o) > H(i) or W(o) > W(i)):
        kh, kw = H(o) // H(i), W(o) // W(i)
        yield (f"upscale:{kh}x{kw}", 2, lambda g, kh=kh, kw=kw: [[v for v in r for _ in range(kw)] for r in g for _ in range(kh)])
    if H(i) % H(o) == 0 and W(i) % W(o) == 0 and (H(i) > H(o) or W(i) > W(o)):
        kh, kw = H(i) // H(o), W(i) // W(o)
        def down(g, kh=kh, kw=kw, mode="mode"):
            if H(g) % kh or W(g) % kw: return None
            bg = bg_of(g); out = []
            for a in range(0, H(g), kh):
                row = []
                for b in range(0, W(g), kw):
                    blk = [v for r in g[a:a + kh] for v in r[b:b + kw]]
                    nz = [v for v in blk if v != bg]
                    row.append(Counter(nz).most_common(1)[0][0] if (mode == "any" and nz) else (Counter(blk).most_common(1)[0][0] if mode == "mode" else bg))
                out.append(row)
            return out
        yield (f"downscale-mode:{kh}x{kw}", 2, down)
        yield (f"downscale-any:{kh}x{kw}", 3, lambda g, d=down: d(g, mode="any"))

def fam_fractal(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i) * H(i), W(i) * W(i)): return
    for inv, bgm in product((False, True), ("mode", "zero")):
        def fn(g, inv=inv, bgm=bgm):
            bg = bg_of(g) if bgm == "mode" else 0
            h, w = H(g), W(g); out = [[bg] * (w * w) for _ in range(h * h)]
            for a in range(h):
                for b in range(w):
                    if (g[a][b] != bg) != inv:
                        for y in range(h):
                            for x in range(w):
                                out[a * h + y][b * w + x] = g[y][x]
            return out
        yield ("fractal" + (":inverse" if inv else "") + ("" if bgm == "mode" else ":bg0"), 3, fn)

SELECTORS = ("largest", "smallest", "most_colours", "unique_colour", "unique_shape", "topmost", "bottommost",
             "leftmost", "rightmost", "densest_bbox", "sparsest_bbox", "most_frequent_shape")

def select(objs, g, how):
    if not objs: return None
    key = {
        "largest": lambda o: len(o), "smallest": lambda o: -len(o),
        "topmost": lambda o: -bbox(o)[0], "bottommost": lambda o: bbox(o)[2],
        "leftmost": lambda o: -bbox(o)[1], "rightmost": lambda o: bbox(o)[3],
        "most_colours": lambda o: len({g[y][x] for y, x in o}),
        "densest_bbox": lambda o: len(o) / ((bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1)),
        "sparsest_bbox": lambda o: -len(o) / ((bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1)),
    }
    if how in key:
        vals = [key[how](o) for o in objs]; best = max(vals)
        hits = [o for o, v in zip(objs, vals) if v == best]
        return hits[0] if len(hits) == 1 else None
    def shape(o):
        r0, c0, _, _ = bbox(o); return tuple(sorted((y - r0, x - c0) for y, x in o))
    if how == "unique_colour":
        cs = Counter(g[o[0][0]][o[0][1]] for o in objs)
        hits = [o for o in objs if cs[g[o[0][0]][o[0][1]]] == 1]
        return hits[0] if len(hits) == 1 else None
    if how in ("unique_shape", "most_frequent_shape"):
        cs = Counter(shape(o) for o in objs)
        if how == "unique_shape":
            hits = [o for o in objs if cs[shape(o)] == 1]
        else:
            top = cs.most_common(1)[0]
            if len(cs) > 1 and cs.most_common(2)[1][1] == top[1]: return None
            hits = [o for o in objs if shape(o) == top[0]][:1]
        return hits[0] if len(hits) == 1 else None
    return None

def fam_crop(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) > H(i) or W(o) > W(i) or (H(o), W(o)) == (H(i), W(i)): return
    def all_fg(g):
        bg = bg_of(g); cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] != bg]
        return crop(g, bbox(cells)) if cells else None
    yield ("crop:all-foreground", 2, all_fg)
    for diag, byc, how in product((False, True), (True, False), SELECTORS):
        def fn(g, diag=diag, byc=byc, how=how):
            bg = bg_of(g); ob = select(objects(g, bg, diag, byc), g, how)
            return crop(g, bbox(ob)) if ob else None
        yield (f"crop:object[{'8' if diag else '4'}{'' if byc else ',multi'}]:{how}", 3, fn)

def split_panels(g):
    """Split by full-length single-colour separator rows/cols; returns (panels, sep_colour) or None."""
    h, w = H(g), W(g)
    for sc in colours(g):
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols: continue
        if len(rows) == h or len(cols) == w: continue
        rb = [-1] + rows + [h]; cb = [-1] + cols + [w]
        panels = []
        for a in range(len(rb) - 1):
            for b in range(len(cb) - 1):
                if rb[a + 1] - rb[a] > 1 and cb[b + 1] - cb[b] > 1:
                    panels.append(crop(g, (rb[a] + 1, cb[b] + 1, rb[a + 1] - 1, cb[b + 1] - 1)))
        if len(panels) >= 2 and all((H(p), W(p)) == (H(panels[0]), W(panels[0])) for p in panels):
            return panels, sc
    # no separator: equal halves
    if h % 2 == 0 and h >= 2: pass
    return None

def halves(g):
    h, w = H(g), W(g); out = []
    if w % 2 == 0: out.append([[r[:w // 2] for r in g], [r[w // 2:] for r in g]])
    if h % 2 == 0: out.append([g[:h // 2], g[h // 2:]])
    return out

BOOLS = {"and": lambda a, b: a and b, "or": lambda a, b: a or b, "xor": lambda a, b: a != b,
         "nor": lambda a, b: not a and not b, "a_not_b": lambda a, b: a and not b, "b_not_a": lambda a, b: b and not a}

def fam_panels(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) == (H(i), W(i)): return
    def getp(g, mode):
        if mode == "sep":
            sp = split_panels(g); return sp[0] if sp else None
        hs = halves(g); k = int(mode[-1])
        return hs[k] if len(hs) > k else None
    oc = [c for c in colours(o)]
    for mode in ("sep", "half0", "half1"):
        ps = getp(i, mode)
        if not ps or (H(ps[0]), W(ps[0])) != (H(o), W(o)): continue
        # boolean combination of two panels -> constant colour on true, bg on false
        if len(ps) == 2:
            for bn, f in BOOLS.items():
                for on_c in oc:
                    def fn(g, mode=mode, f=f, on_c=on_c):
                        p = getp(g, mode)
                        if not p or len(p) != 2: return None
                        a, b = p; bg = bg_of(g)
                        return [[on_c if f(a[y][x] != bg, b[y][x] != bg) else bg for x in range(W(a))] for y in range(H(a))]
                    yield (f"panels[{mode}]:{bn}->colour", 3, fn)
        # overlay: later panels painted over earlier (non-zero wins), both orders
        for order in ("fwd", "rev"):
            def ov(g, mode=mode, order=order):
                p = getp(g, mode)
                if not p: return None
                p = p if order == "fwd" else p[::-1]
                bgg = bg_of(g); out = [r[:] for r in p[0]]
                for q in p[1:]:
                    for y in range(H(q)):
                        for x in range(W(q)):
                            if q[y][x] != bgg: out[y][x] = q[y][x]
                return out
            yield (f"panels[{mode}]:overlay-{order}", 3, ov)
        for how in ("most_fg", "least_fg", "unique", "most_colours", "least_colours"):
            def sel(g, mode=mode, how=how):
                p = getp(g, mode)
                if not p: return None
                if how == "unique":
                    cs = Counter(str(q) for q in p); hits = [q for q in p if cs[str(q)] == 1]
                    return hits[0] if len(hits) == 1 else None
                bgg = bg_of(g)
                key = {"most_fg": lambda q: sum(v != bgg for r in q for v in r), "least_fg": lambda q: -sum(v != bgg for r in q for v in r),
                       "most_colours": lambda q: len(colours(q)), "least_colours": lambda q: -len(colours(q))}[how]
                vals = [key(q) for q in p]; hits = [q for q, v in zip(p, vals) if v == max(vals)]
                return hits[0] if len(hits) == 1 else None
            yield (f"panels[{mode}]:select-{how}", 3, sel)

SYMS = {"fh": fh, "fv": fv, "r180": r180, "T": T, "aT": at, "r90": r90}

def fam_symmetry(train):
    """Occluder colour (present in input, absent in output) is replaced by symmetric counterparts.
    Output either the repaired grid (same size) or the repaired patch under the occluder."""
    i, o = train[0]["input"], train[0]["output"]
    occ = [c for c in colours(i) if all(c in colours(p["input"]) and c not in colours(p["output"]) for p in train)]
    same = (H(o), W(o)) == (H(i), W(i))
    for c in occ:
        for combo in (("fh",), ("fv",), ("fh", "fv"), ("fh", "fv", "r180"), ("T",), ("fh", "fv", "T", "aT", "r90", "r180")):
            def rep(g, c=c, combo=combo):
                out = [r[:] for r in g]
                for _ in range(3):
                    changed = False
                    for k in combo:
                        if k in ("T", "aT", "r90") and H(g) != W(g): return None
                        m = SYMS[k](out)
                        for y in range(H(g)):
                            for x in range(W(g)):
                                if out[y][x] == c and m[y][x] != c:
                                    out[y][x] = m[y][x]; changed = True
                    if not changed: break
                return None if any(v == c for r in out for v in r) else out
            if same:
                yield (f"symmetry-repair:{'+'.join(combo)}", 3, rep)
            else:
                def patch(g, c=c, rep=rep):
                    cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == c]
                    if not cells: return None
                    r = rep(g)
                    return crop(r, bbox(cells)) if r else None
                yield (f"symmetry-patch:{'+'.join(combo)}", 4, patch)

def fam_gravity(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    # (to frame, from frame): transform so that the motion direction becomes 'down'
    frames = {"down": (lambda g: g, lambda g: g), "up": (fv, fv), "right": (T, T), "left": (lambda g: T(fh(g)), lambda g: fh(T(g)))}
    for d, (to, back) in frames.items():
        def fn(g, to=to, back=back):
            bg = bg_of(g); out = [r[:] for r in to(g)]
            for x in range(W(out)):
                col = [out[y][x] for y in range(H(out)) if out[y][x] != bg]
                for y in range(H(out)): out[y][x] = bg
                for k, v in enumerate(reversed(col)): out[H(out) - 1 - k][x] = v
            return back(out)
        yield (f"gravity-cells:{d}", 2, fn)

def fam_colour_count(train):
    """Output is a 1-row/col strip or square filled from colour counts (sorted by frequency)."""
    i, o = train[0]["input"], train[0]["output"]
    if H(o) * W(o) > 30: return
    def strip(g, orient):
        bg = bg_of(g); cs = Counter(v for r in g for v in r if v != bg)
        seq = [c for c, _ in cs.most_common()]
        if not seq: return None
        return [seq] if orient == "row" else [[c] for c in seq]
    yield ("colours-by-frequency:row", 3, lambda g: strip(g, "row"))
    yield ("colours-by-frequency:col", 3, lambda g: strip(g, "col"))

FAMILIES = (fam_geometric, fam_tile, fam_scale, fam_fractal, fam_crop, fam_panels, fam_symmetry, fam_gravity, fam_colour_count)


# ------------------------------------------------------------------ cycle 2 families
def fam_crop_colour(train):
    """Crop to the bbox of one colour's cells, or to the interior of that bbox (frame contents).
    The colour is induced: it must reproduce every training output."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) == (H(i), W(i)) or H(o) > H(i) or W(o) > W(i): return
    for c in sorted(colours(i)):
        for inner in (False, True):
            def fn(g, c=c, inner=inner):
                cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == c]
                if not cells: return None
                r0, c0, r1, c1 = bbox(cells)
                if inner: r0, c0, r1, c1 = r0 + 1, c0 + 1, r1 - 1, c1 - 1
                if r1 < r0 or c1 < c0: return None
                return crop(g, (r0, c0, r1, c1))
            yield (f"crop:colour-bbox{'-interior' if inner else ''}[c{c}]", 3, fn)

def fam_fill_enclosed(train):
    """Background regions not connected to the border are filled with an induced colour."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    new = Counter(o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x])
    for c, _ in new.most_common(2):
        for diag in (False, True):
            def fn(g, c=c, diag=diag):
                bg = bg_of(g); h, w = H(g), W(g); out = [r[:] for r in g]
                seen = [[False] * w for _ in range(h)]
                st = [(y, x) for y in range(h) for x in range(w) if (y in (0, h - 1) or x in (0, w - 1)) and g[y][x] == bg]
                for y, x in st: seen[y][x] = True
                nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
                while st:
                    y, x = st.pop()
                    for dy, dx in nb:
                        yy, xx = y + dy, x + dx
                        if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and g[yy][xx] == bg:
                            seen[yy][xx] = True; st.append((yy, xx))
                for y in range(h):
                    for x in range(w):
                        if g[y][x] == bg and not seen[y][x]: out[y][x] = c
                return out
            yield (f"fill-enclosed[c{c}{',8' if diag else ''}]", 2, fn)

def fam_symmetrize(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for combo in (("fh",), ("fv",), ("fh", "fv", "r180"), ("T",), ("fh", "fv", "T", "aT", "r90", "r180", "r270")):
        def fn(g, combo=combo):
            bg = bg_of(g); out = [r[:] for r in g]
            for k in combo:
                if k in ("T", "aT", "r90", "r270") and H(g) != W(g): return None
                m = (SYMS.get(k) or D8[k])(g)
                for y in range(H(g)):
                    for x in range(W(g)):
                        if out[y][x] == bg and m[y][x] != bg: out[y][x] = m[y][x]
            return out
        yield (f"symmetrize:{'+'.join(combo)}", 2, fn)

def fam_object_filter(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for diag, byc in product((False, True), (True, False)):
        for k in (1, 2, 3):
            def noise(g, k=k, diag=diag, byc=byc):
                bg = bg_of(g); out = [r[:] for r in g]
                for ob in objects(g, bg, diag, byc):
                    if len(ob) <= k:
                        for y, x in ob: out[y][x] = bg
                return out
            yield (f"remove-objects-size<={k}[{'8' if diag else '4'}{'' if byc else ',multi'}]", 2, noise)
        for how in ("largest", "smallest", "unique_colour", "unique_shape", "most_colours"):
            def keep(g, how=how, diag=diag, byc=byc):
                bg = bg_of(g); obs = objects(g, bg, diag, byc); ob = select(obs, g, how)
                if not ob: return None
                out = [[bg] * W(g) for _ in range(H(g))]
                for y, x in ob: out[y][x] = g[y][x]
                return out
            yield (f"keep-object[{'8' if diag else '4'}{'' if byc else ',multi'}]:{how}", 3, keep)

RAYS = {"orth": [(-1, 0), (1, 0), (0, -1), (0, 1)], "diag": [(-1, -1), (-1, 1), (1, -1), (1, 1)],
        "up": [(-1, 0)], "down": [(1, 0)], "left": [(0, -1)], "right": [(0, 1)], "vert": [(-1, 0), (1, 0)], "horiz": [(0, -1), (0, 1)]}

def fam_rays(train):
    """Cells of an induced colour emit rays (own colour) in induced directions, stopping at non-background
    (or passing over it)."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    src = sorted({i[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != bg_of(i)})
    for c in src:
        for dn, dirs in RAYS.items():
            for stop in (True, False):
                def fn(g, c=c, dirs=dirs, stop=stop):
                    bg = bg_of(g); out = [r[:] for r in g]
                    for y in range(H(g)):
                        for x in range(W(g)):
                            if g[y][x] != c: continue
                            for dy, dx in dirs:
                                yy, xx = y + dy, x + dx
                                while 0 <= yy < H(g) and 0 <= xx < W(g):
                                    if g[yy][xx] != bg:
                                        if stop: break
                                    else: out[yy][xx] = c
                                    yy += dy; xx += dx
                    return out
                yield (f"rays[c{c}]:{dn}{':stop' if stop else ':through'}", 3, fn)

def fam_connect(train):
    """Same-colour cells sharing a row/column are joined by a line of their colour (or an induced colour)."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    new = [None] + [c for c, _ in Counter(o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x]).most_common(1)]
    for axes in ("both", "row", "col"):
        for lc in new:
            def fn(g, axes=axes, lc=lc):
                bg = bg_of(g); out = [r[:] for r in g]
                pts = {}
                for y in range(H(g)):
                    for x in range(W(g)):
                        if g[y][x] != bg: pts.setdefault(g[y][x], []).append((y, x))
                for c, ps in pts.items():
                    for (y1, x1), (y2, x2) in product(ps, ps):
                        if (y1, x1) >= (y2, x2): continue
                        if y1 == y2 and axes in ("both", "row"):
                            for x in range(min(x1, x2) + 1, max(x1, x2)):
                                if out[y1][x] == bg: out[y1][x] = lc if lc is not None else c
                        if x1 == x2 and axes in ("both", "col"):
                            for y in range(min(y1, y2) + 1, max(y1, y2)):
                                if out[y][x1] == bg: out[y][x1] = lc if lc is not None else c
                return out
            yield (f"connect-same-colour:{axes}{'' if lc is None else f'[c{lc}]'}", 3, fn)
    for lc in new:
        def dfn(g, lc=lc):
            bg = bg_of(g); out = [r[:] for r in g]; pts = {}
            for y in range(H(g)):
                for x in range(W(g)):
                    if g[y][x] != bg: pts.setdefault(g[y][x], []).append((y, x))
            for c, ps in pts.items():
                for (y1, x1), (y2, x2) in product(ps, ps):
                    if (y1, x1) >= (y2, x2) or abs(y1 - y2) != abs(x1 - x2): continue
                    sy = 1 if y2 > y1 else -1; sx = 1 if x2 > x1 else -1
                    for k in range(1, abs(y2 - y1)):
                        if out[y1 + k * sy][x1 + k * sx] == bg: out[y1 + k * sy][x1 + k * sx] = lc if lc is not None else c
            return out
        yield (f"connect-same-colour:diagonal{'' if lc is None else f'[c{lc}]'}", 3, dfn)

PROPS = {"size": lambda o, g: len(o), "height": lambda o, g: bbox(o)[2] - bbox(o)[0] + 1, "width": lambda o, g: bbox(o)[3] - bbox(o)[1] + 1,
         "size_rank": None, "is_rect": lambda o, g: len(o) == (bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1),
         "shape": lambda o, g: tuple(sorted((y - bbox(o)[0], x - bbox(o)[1]) for y, x in o)),
         "touches_border": lambda o, g: any(y in (0, H(g) - 1) or x in (0, W(g) - 1) for y, x in o),
         "n_holes": None}

def fam_recolour_by_property(train):
    """Each object is recoloured by a mapping property-value -> colour induced from training pairs.
    Refuses property values never seen in training."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for diag, byc in product((False, True), (True, False)):
        for pn in ("size", "height", "width", "size_rank", "is_rect", "shape", "touches_border", "n_holes", "adj_colours",
                   "colour_set", "shape_count", "extreme_size", "area_rank"):
            def pval(obs, g, pn=pn):
                if pn == "size_rank":
                    sizes = sorted({len(x) for x in obs}, reverse=True)
                    return [sizes.index(len(x)) for x in obs]
                if pn == "area_rank":
                    ar = lambda o: (bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1)
                    vals = sorted({ar(x) for x in obs}, reverse=True)
                    return [vals.index(ar(x)) for x in obs]
                if pn == "extreme_size":
                    mx = max(len(x) for x in obs); mn = min(len(x) for x in obs)
                    return ["max" if len(x) == mx else "min" if len(x) == mn else "mid" for x in obs]
                if pn == "shape_count":
                    sh = [PROPS["shape"](x, g) for x in obs]; cs = Counter(sh)
                    return [min(cs[s], 3) for s in sh]
                if pn == "colour_set":
                    return [frozenset(g[y][x] for y, x in o) for o in obs]
                if pn == "adj_colours":
                    bg = bg_of(g); out = []
                    for o in obs:
                        cs = set(o); col = set()
                        for y, x in o:
                            for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                                yy, xx = y + dy, x + dx
                                if 0 <= yy < H(g) and 0 <= xx < W(g) and (yy, xx) not in cs and g[yy][xx] != bg: col.add(g[yy][xx])
                        out.append(frozenset(col))
                    return out
                if pn == "n_holes":
                    bg = bg_of(g); out = []
                    for o in obs:
                        r0, c0, r1, c1 = bbox(o); cs = set(o)
                        sub = [[1 if (y, x) in cs else 0 for x in range(c0 - 1, c1 + 2)] for y in range(r0 - 1, r1 + 2)]
                        holes = objects(sub, 1, False, True)   # regions of 0 (non-object) cells
                        out.append(sum(1 for hcells in holes if not any(y in (0, len(sub) - 1) or x in (0, len(sub[0]) - 1) for y, x in hcells)))
                    return out
                return [PROPS[pn](x, g) for x in obs]
            m = {}; ok = True
            for p in train:
                gi, go = p["input"], p["output"]
                if (H(gi), W(gi)) != (H(go), W(go)): ok = False; break
                bg = bg_of(gi); obs = objects(gi, bg, diag, byc)
                for ob, v in zip(obs, pval(obs, gi)):
                    cs = {go[y][x] for y, x in ob}
                    if len(cs) != 1 or m.setdefault(v, cs.pop()) != go[ob[0][0]][ob[0][1]]: ok = False; break
                if not ok: break
            if not ok or not m: continue
            def fn(g, m=m, diag=diag, byc=byc, pval=pval):
                bg = bg_of(g); out = [r[:] for r in g]; obs = objects(g, bg, diag, byc)
                for ob, v in zip(obs, pval(obs, g)):
                    if v not in m: return None
                    for y, x in ob: out[y][x] = m[v]
                return out
            yield (f"recolour-by-{pn}[{'8' if diag else '4'}{'' if byc else ',multi'}]", 3, fn)

def fam_panels_multi(train):
    i, o = train[0]["input"], train[0]["output"]
    sp = split_panels(i)
    if not sp or len(sp[0]) < 3 or (H(sp[0][0]), W(sp[0][0])) != (H(o), W(o)): return
    for rule in ("all", "any", "one", "none"):
        for on_c in sorted(colours(o)):
            def fn(g, rule=rule, on_c=on_c):
                s = split_panels(g)
                if not s: return None
                ps = s[0]; bg = bg_of(g)
                if any((H(p), W(p)) != (H(ps[0]), W(ps[0])) for p in ps): return None
                def f(y, x):
                    n = sum(p[y][x] != bg for p in ps)
                    return {"all": n == len(ps), "any": n > 0, "one": n == 1, "none": n == 0}[rule]
                return [[on_c if f(y, x) else bg for x in range(W(ps[0]))] for y in range(H(ps[0]))]
            yield (f"panels[sep]:{rule}-of-{len(sp[0])}->colour", 3, fn)

RESHAPERS = ("tile", "crop", "panels", "upscale", "downscale", "fractal", "symmetry-patch", "symmetry-offset-patch", "kronecker")

def compose_dihedral(base):
    """Any output-reshaping program followed by a dihedral transform (one extra step)."""
    for name, cost, fn in base:
        if not name.startswith(RESHAPERS): continue
        for k, f in D8.items():
            if k == "id": continue
            yield (name + ">" + k, cost + 1, lambda g, fn=fn, f=f: (lambda r: f(r) if r else None)(fn(g)))

def fam_codex(train):
    try:
        import codex_ops
    except Exception:
        return
    yield from codex_ops.programs(train)

FAMILIES = FAMILIES + (fam_crop_colour, fam_fill_enclosed, fam_symmetrize, fam_object_filter, fam_rays, fam_connect, fam_recolour_by_property, fam_panels_multi, fam_codex)

# ------------------------------------------------------------------ cycle 9 families (motivated by training + eval half A)
def _sym_transforms(h, w):
    """Candidate symmetries with arbitrary axis/centre positions: (name, f(y,x)->(y,x))."""
    out = []
    for s in range(w // 2, w + w // 2):           # horizontal mirror x -> s - x  (axis at s/2)
        out.append((f"mirror-x@{s}", lambda y, x, s=s: (y, s - x)))
    for s in range(h // 2, h + h // 2):
        out.append((f"mirror-y@{s}", lambda y, x, s=s: (s - y, x)))
    for d in range(-3, 4):                        # transpose about an offset main diagonal
        out.append((f"transpose@{d}", lambda y, x, d=d: (x - d, y + d)))
    return out

def infer_symmetries(g, occ, min_agree=0.98, min_pairs=0.3):
    h, w = H(g), W(g); acc = []
    for name, f in _sym_transforms(h, w):
        agree = tot = 0
        for y in range(h):
            for x in range(w):
                if g[y][x] == occ: continue
                yy, xx = f(y, x)
                if 0 <= yy < h and 0 <= xx < w and g[yy][xx] != occ:
                    tot += 1; agree += g[y][x] == g[yy][xx]
        if tot >= min_pairs * h * w and agree >= min_agree * tot:
            acc.append((name, f))
    return acc

def offset_repair(g, occ):
    syms = infer_symmetries(g, occ)
    if not syms: return None
    out = [r[:] for r in g]; h, w = H(g), W(g)
    for _ in range(4):
        changed = False
        for y in range(h):
            for x in range(w):
                if out[y][x] != occ: continue
                for _, f in syms:
                    yy, xx = f(y, x)
                    if 0 <= yy < h and 0 <= xx < w and out[yy][xx] != occ:
                        out[y][x] = out[yy][xx]; changed = True; break
        if not changed: break
    return None if any(v == occ for r in out for v in r) else out

def fam_symmetry_offset(train):
    i, o = train[0]["input"], train[0]["output"]
    occ = [c for c in colours(i) if all(c in colours(p["input"]) and c not in colours(p["output"]) for p in train)]
    same = (H(o), W(o)) == (H(i), W(i))
    for c in occ:
        if same:
            yield (f"symmetry-offset-repair[c{c}]", 4, lambda g, c=c: offset_repair(g, c))
        else:
            def patch(g, c=c):
                cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == c]
                if not cells: return None
                r = offset_repair(g, c)
                return crop(r, bbox(cells)) if r else None
            yield (f"symmetry-offset-patch[c{c}]", 4, patch)

def fam_object_symmetrize(train):
    """Each object keeps only cells whose mirror partner (about the object's best axis) is present."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for axis in ("x", "y"):
        for diag, byc in product((True, False), (True, False)):
            def fn(g, axis=axis, diag=diag, byc=byc):
                bg = bg_of(g); out = [r[:] for r in g]
                for ob in objects(g, bg, diag, byc):
                    cells = set(ob); r0, c0, r1, c1 = bbox(ob); best = None
                    lo, hi = (2 * c0, 2 * c1) if axis == "x" else (2 * r0, 2 * r1)
                    for s in range(lo, hi + 1):
                        keep = {(y, x) for y, x in cells if ((y, s - x) if axis == "x" else (s - y, x)) in cells and
                                g[y][x] == g[(y if axis == "x" else s - y)][(s - x if axis == "x" else x)]}
                        if best is None or len(keep) > len(best): best = keep
                    for y, x in cells - (best or set()): out[y][x] = bg
                return out
            yield (f"object-symmetrize:{axis}[{'8' if diag else '4'}{'' if byc else ',multi'}]", 3, fn)

def fam_kronecker(train):
    """Two objects: a single-colour mask and a pattern tile; output places the tile at each (non-)mask cell."""
    i, o = train[0]["input"], train[0]["output"]
    for inv in (False, True):
        for pick in ("mono_mask", "small_mask"):
            def fn(g, inv=inv, pick=pick):
                bg = bg_of(g); obs = objects(g, bg, True, False)
                if len(obs) != 2: return None
                a, b = [crop(g, bbox(x)) for x in obs]
                if pick == "mono_mask":
                    ma = len(colours(a) - {bg}) == 1; mb = len(colours(b) - {bg}) == 1
                    if ma == mb: return None
                    mask, tile = (a, b) if ma else (b, a)
                else:
                    mask, tile = (a, b) if H(a) * W(a) <= H(b) * W(b) else (b, a)
                out = []
                for y in range(H(mask)):
                    blk = [tile if ((mask[y][x] != bg) != inv) else [[bg] * W(tile) for _ in range(H(tile))] for x in range(W(mask))]
                    for rr in range(H(tile)): out.append(sum((q[rr] for q in blk), []))
                return out
            yield (f"kronecker:{pick}{':inverse' if inv else ''}", 3, fn)

def frame_interior(g):
    """Interior of the largest rectangle whose border cells all share one non-background colour."""
    bg = bg_of(g); best = None
    for ob in objects(g, bg, False, True):
        r0, c0, r1, c1 = bbox(ob)
        if r1 - r0 < 2 or c1 - c0 < 2: continue
        col = g[ob[0][0]][ob[0][1]]
        border = [(r0, x) for x in range(c0, c1 + 1)] + [(r1, x) for x in range(c0, c1 + 1)] + [(y, c0) for y in range(r0, r1 + 1)] + [(y, c1) for y in range(r0, r1 + 1)]
        if all(g[y][x] == col for y, x in border):
            area = (r1 - r0) * (c1 - c0)
            if best is None or area > best[0]: best = (area, (r0 + 1, c0 + 1, r1 - 1, c1 - 1))
    return crop(g, best[1]) if best else None

def fam_frame_crop(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) >= H(i) and W(o) >= W(i): return
    yield ("crop:frame-interior", 2, frame_interior)

def fam_outline(train):
    """Background cells adjacent (4 or 8) to objects of an induced colour are painted an induced colour."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    new = Counter(o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x])
    srcs = sorted(colours(i) - {bg_of(i)})
    for c, _ in new.most_common(2):
        for s in srcs + [None]:
            for diag in (False, True):
                def fn(g, c=c, s=s, diag=diag):
                    bg = bg_of(g); out = [r[:] for r in g]
                    nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
                    for y in range(H(g)):
                        for x in range(W(g)):
                            if g[y][x] != bg: continue
                            for dy, dx in nb:
                                yy, xx = y + dy, x + dx
                                if 0 <= yy < H(g) and 0 <= xx < W(g) and g[yy][xx] != bg and (s is None or g[yy][xx] == s):
                                    out[y][x] = c; break
                    return out
                yield (f"outline[c{c}{'' if s is None else f',src{s}'}{',8' if diag else ''}]", 3, fn)

def fam_period_extend(train):
    """Rows (or columns) containing a partial periodic sequence are continued periodically across the grid."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for axis in ("row", "col"):
        for mode in ("from_first", "full"):
            def fn(g, axis=axis, mode=mode):
                gg = g if axis == "row" else T(g); bg = bg_of(g); out = [r[:] for r in gg]
                for y, row in enumerate(gg):
                    idx = [x for x, v in enumerate(row) if v != bg]
                    if len(idx) < 2: continue
                    a, b = idx[0], idx[-1]; seg = row[a:b + 1]
                    for p in range(1, len(seg) + 1):   # smallest period consistent with the observed segment
                        if all(seg[k] == seg[k % p] for k in range(len(seg))) and p < len(seg): break
                    else:
                        continue
                    start = a if mode == "from_first" else a % p
                    for x in range(start, len(row)):
                        out[y][x] = seg[(x - a) % p]
                return out if axis == "row" else T(out)
            yield (f"period-extend:{axis}:{mode}", 3, fn)
        def prog(g, axis=axis):
            """First two marks set the step; marks repeat up to the last mark, in the last mark's colour."""
            gg = g if axis == "row" else T(g); bg = bg_of(g); out = [r[:] for r in gg]
            for y, row in enumerate(gg):
                idx = [x for x, v in enumerate(row) if v != bg]
                if len(idx) < 3: continue
                a, b, last = idx[0], idx[1], idx[-1]; p = b - a
                if p <= 0 or (last - a) % p: continue
                for x in range(a, last + 1, p): out[y][x] = row[last]
            return out if axis == "row" else T(out)
        yield (f"progression:{axis}", 3, prog)
    def prog_row(row, bg):
        """Two same-colour marks set a step; continue to the odd-coloured last mark (in its colour), else to the edge."""
        idx = [x for x, v in enumerate(row) if v != bg]
        if len(idx) < 2 or row[idx[0]] != row[idx[1]]: return None
        a, p = idx[0], idx[1] - idx[0]
        rest = [x for x in idx[2:] if row[x] != row[a]]
        if any(row[x] == row[a] for x in idx[2:]): return None
        if len(rest) > 1: return None
        keep_other = False
        if rest and (rest[0] - a) % p == 0:
            end, col = rest[0], row[rest[0]]
        else:                                   # odd mark off the lattice (or none): continue to the edge
            end, col, keep_other = len(row) - 1, row[a], bool(rest)
        new = list(row)
        for x in range(a, end + 1, p):
            if keep_other and row[x] != bg and row[x] != row[a]: continue
            new[x] = col
        return new
    def prog_row_bi(row, bg):
        """The anchoring same-colour pair sits at a grid edge; progress away from that edge."""
        if row and row[0] != bg:
            return prog_row(row, bg)
        if row and row[-1] != bg:
            r = prog_row(row[::-1], bg)
            return r[::-1] if r is not None else None
        return None
    def prog_auto(g):
        bg = bg_of(g)
        cand = {}
        for ax, gg in (("r", g), ("c", T(g))):
            rows = [prog_row_bi(r, bg) for r in gg]
            cand[ax] = (sum(1 for r in rows if r is not None), gg, rows)
        (nr, _, _), (nc, _, _) = cand["r"], cand["c"]
        if nr == nc: return None
        ax = "r" if nr > nc else "c"; _, gg, rows = cand[ax]
        out = [r if r is not None else list(o) for r, o in zip(rows, gg)]
        return out if ax == "r" else T(out)
    yield ("progression:auto-axis", 4, prog_auto)

FAMILIES = FAMILIES + (fam_symmetry_offset, fam_object_symmetrize, fam_kronecker, fam_frame_crop, fam_outline, fam_period_extend)

# ------------------------------------------------------------------ cycle 10b families
def key_pairs(g, bg):
    """In-grid colour key: isolated 2-cell objects made of two different colours define a->b."""
    m = {}; keycells = set()
    for ob in objects(g, bg, False, False):
        if len(ob) != 2: continue
        (y1, x1), (y2, x2) = sorted(ob)
        a, b = g[y1][x1], g[y2][x2]
        if a == b: continue
        if m.get(a, b) != b: return None, None
        m[a] = b; keycells |= set(ob)
    return (m, keycells) if m else (None, None)

def fam_key_recolour(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for rev in (False, True):
        def fn(g, rev=rev):
            bg = bg_of(g); m, kc = key_pairs(g, bg)
            if not m: return None
            if rev: m = {v: k for k, v in m.items()}
            return [[m.get(v, v) if (y, x) not in kc else v for x, v in enumerate(r)] for y, r in enumerate(g)]
        yield (f"key-recolour{':reverse' if rev else ''}", 3, fn)

def fam_marker_object(train):
    """Crop / keep the (multi-colour) object that contains a cell of an induced marker colour."""
    i, o = train[0]["input"], train[0]["output"]
    same = (H(o), W(o)) == (H(i), W(i))
    for c in sorted(colours(i) - {bg_of(i)}):
        for diag in (True, False):
            def pick(g, c=c, diag=diag):
                bg = bg_of(g); hits = [ob for ob in objects(g, bg, diag, False) if any(g[y][x] == c for y, x in ob)]
                return hits[0] if len(hits) == 1 else None
            if same:
                def keep(g, pick=pick):
                    ob = pick(g)
                    if not ob: return None
                    bg = bg_of(g); out = [[bg] * W(g) for _ in range(H(g))]
                    for y, x in ob: out[y][x] = g[y][x]
                    return out
                yield (f"keep-object-with[c{c}{',8' if diag else ''}]", 3, keep)
            else:
                yield (f"crop:object-with[c{c}{',8' if diag else ''}]", 3, lambda g, pick=pick: (lambda ob: crop(g, bbox(ob)) if ob else None)(pick(g)))

def fam_count_output(train):
    """Output size equals a count of objects; filled with an induced colour or the counted colour."""
    shapes = {(H(p["output"]), W(p["output"])) for p in train}
    outs_c = [colours(p["output"]) for p in train]
    if not all(len(c) == 1 for c in outs_c): return
    for diag, byc in product((False, True), (True, False)):
        for orient in ("row", "col"):
            for colour_rule in ("most_common_object_colour", "constant"):
                const = list(outs_c[0])[0]
                def fn(g, diag=diag, byc=byc, orient=orient, colour_rule=colour_rule, const=const):
                    bg = bg_of(g); obs = objects(g, bg, diag, byc)
                    if not obs: return None
                    col = const if colour_rule == "constant" else Counter(g[o[0][0]][o[0][1]] for o in obs).most_common(1)[0][0]
                    n = len(obs) if colour_rule == "constant" else sum(1 for o in obs if g[o[0][0]][o[0][1]] == col)
                    return [[col] * n] if orient == "row" else [[col] for _ in range(n)]
                yield (f"count-objects[{'8' if diag else '4'}{'' if byc else ',multi'}]:{orient}:{colour_rule}", 3, fn)

def fam_colour_select_fill(train):
    """Output has a fixed shape (same in all pairs) filled with one colour chosen by a rule."""
    shapes = {(H(p["output"]), W(p["output"])) for p in train}
    if len(shapes) != 1 or not all(len(colours(p["output"])) == 1 for p in train): return
    h, w = shapes.pop()
    rules = {"most_frequent_fg": lambda g, bg: Counter(v for r in g for v in r if v != bg).most_common(1)[0][0],
             "least_frequent_fg": lambda g, bg: Counter(v for r in g for v in r if v != bg).most_common()[-1][0],
             "largest_object": lambda g, bg: (lambda ob: g[ob[0][0]][ob[0][1]])(max(objects(g, bg, False, True), key=len)),
             "smallest_object": lambda g, bg: (lambda ob: g[ob[0][0]][ob[0][1]])(min(objects(g, bg, False, True), key=len)),
             "most_objects": lambda g, bg: Counter(g[o[0][0]][o[0][1]] for o in objects(g, bg, False, True)).most_common(1)[0][0]}
    for rn, f in rules.items():
        def fn(g, f=f):
            bg = bg_of(g); c = f(g, bg)
            return [[c] * w for _ in range(h)]
        yield (f"fill-{h}x{w}-with:{rn}", 3, fn)

FAMILIES = FAMILIES + (fam_key_recolour, fam_marker_object, fam_count_output, fam_colour_select_fill)

def candidates(train):
    out = []
    for fam in FAMILIES:
        try:
            out.extend(fam(train))
        except Exception:
            continue
    out = out + list(compose_dihedral(out))
    out.sort(key=lambda x: x[1])
    return out

def run(fn, g):
    try:
        r = fn(g)
        if r is not None and not isinstance(r, list): r = [list(map(int, row)) for row in r]
        elif r is not None: r = [list(map(int, row)) for row in r]
        if r is None or not r or not r[0] or H(r) > 30 or W(r) > 30: return None
        return r
    except Exception:
        return None

# ------------------------------------------------------------------ cycle 11 families: object motion
DIRV = {"down": (1, 0), "up": (-1, 0), "left": (0, -1), "right": (0, 1)}

def static_colours(train):
    """Colours whose cells never change between input and output in any training pair (obstacles/anchors)."""
    st = None
    for p in train:
        i, o = p["input"], p["output"]
        if (H(i), W(i)) != (H(o), W(o)): return set()
        cs = {v for r in i for v in r}
        ok = {c for c in cs if all(o[y][x] == c for y in range(H(i)) for x in range(W(i)) if i[y][x] == c)}
        st = ok if st is None else st & ok
    return st or set()

def slide(g, movers, statics, dy, dx, bg):
    """Move each mover object rigidly by (dy,dx) steps until the next step would leave the grid or hit
    a non-background cell that is not part of itself. Movers are processed from the leading edge."""
    out = [r[:] for r in g]
    key = lambda ob: -max(y * dy + x * dx for y, x in ob)
    for ob in sorted(movers, key=key):
        col = {(y, x): out[y][x] for y, x in ob}
        for y, x in ob: out[y][x] = bg
        cur = list(ob); k = 0
        while True:
            nxt = [(y + dy, x + dx) for y, x in cur]
            if any(not (0 <= y < H(g) and 0 <= x < W(g)) or out[y][x] != bg for y, x in nxt): break
            cur = nxt; k += 1
        for (y0, x0), (y, x) in zip(ob, cur): out[y][x] = col[(y0, x0)]
    return out

def fam_object_gravity(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    st = static_colours(train)
    for d, (dy, dx) in DIRV.items():
        for diag, byc in product((False, True), (True, False)):
            def fn(g, dy=dy, dx=dx, diag=diag, byc=byc):
                bg = bg_of(g); obs = objects(g, bg, diag, byc)
                movers = [ob for ob in obs if not all(g[y][x] in st for y, x in ob)]
                if not movers or len(movers) == len(obs) and st: return None
                return slide(g, movers, st, dy, dx, bg)
            yield (f"object-gravity:{d}[{'8' if diag else '4'}{'' if byc else ',multi'}]", 3, fn)

def fam_move_to_target(train):
    """Each mover object moves in a straight line toward the static (anchor) object it faces, stopping on contact."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    st = static_colours(train)
    if not st: return
    for diag, byc in product((False, True), (True, False)):
        def fn(g, diag=diag, byc=byc):
            bg = bg_of(g); obs = objects(g, bg, diag, byc)
            anchors = [ob for ob in obs if all(g[y][x] in st for y, x in ob)]
            movers = [ob for ob in obs if ob not in anchors]
            if not anchors or not movers: return None
            out = [r[:] for r in g]
            for ob in movers:
                r0, c0, r1, c1 = bbox(ob); best = None
                for an in anchors:
                    a0, b0, a1, b1 = bbox(an)
                    if b0 <= c1 and c0 <= b1:           # vertical overlap -> move up/down
                        d = (1, 0) if a0 > r1 else (-1, 0) if a1 < r0 else None
                        dist = a0 - r1 if a0 > r1 else r0 - a1
                    elif a0 <= r1 and r0 <= a1:         # horizontal overlap -> move left/right
                        d = (0, 1) if b0 > c1 else (0, -1) if b1 < c0 else None
                        dist = b0 - c1 if b0 > c1 else c0 - b1
                    else:
                        continue
                    if d and (best is None or dist < best[0]): best = (dist, d)
                if best: out = slide(out, [ob], st, best[1][0], best[1][1], bg)
            return out
        yield (f"move-to-anchor[{'8' if diag else '4'}{'' if byc else ',multi'}]", 3, fn)

FAMILIES = FAMILIES + (fam_object_gravity, fam_move_to_target)

def residual_steps(train, k=4):
    """Same-size first steps that move every pair strictly closer to its output and never break a correct cell."""
    outs = [p["output"] for p in train]
    if any((H(p["input"]), W(p["input"])) != (H(o), W(o)) for p, o in zip(train, outs)): return []
    scored = []
    for name, cost, fn in candidates(train):
        if name.startswith(RESHAPERS) or ">" in name: continue
        gain = 0; ok = True; preds = []
        for p, o in zip(train, outs):
            i = p["input"]; r = run(fn, i)
            if r is None or (H(r), W(r)) != (H(o), W(o)) or r == i: ok = False; break
            broke = any(i[y][x] == o[y][x] and r[y][x] != o[y][x] for y in range(H(o)) for x in range(W(o)))
            fixed = sum(i[y][x] != o[y][x] and r[y][x] == o[y][x] for y in range(H(o)) for x in range(W(o)))
            if broke or fixed == 0: ok = False; break
            gain += fixed; preds.append(r)
        if ok: scored.append((-gain, cost, name, fn, preds))
    scored.sort(key=lambda x: (x[0], x[1]))
    return scored[:k]

def search2(task, max_programs=4):
    """Two-step programs: a residual first step, then a full single-step search on its outputs."""
    res = []
    for _, cost, name, fn, preds in residual_steps(task["train"]):
        sub = {"train": [{"input": r, "output": p["output"]} for r, p in zip(preds, task["train"])],
               "test": [{"input": run(fn, t["input"])} for t in task["test"]]}
        if any(t["input"] is None for t in sub["test"]): continue
        for r in search(sub, max_programs=2, allow2=False):
            res.append({"program": name + " ; " + r["program"], "cost": cost + r["cost"], "preds": r["preds"]})
        if len(res) >= max_programs: break
    return res

def search(task, max_programs=6, allow2=True):
    """Return verified programs: list of dicts {program, cost, preds}. Cheapest first; with and without colour map."""
    train = task["train"]; outs = [p["output"] for p in train]
    found = []
    # identity + colour map alone (pure recolouring)
    ident = [p["input"] for p in train]
    m = fit_cmap(ident, outs)
    if m and any(k != v for k, v in m.items()) and all(apply_cmap(a, m) == b for a, b in zip(ident, outs)):
        found.append(("colour-map", 1, lambda g, m=m: apply_cmap(g, m)))
    for name, cost, fn in candidates(train):
        preds = []
        for p, o in zip(train, outs):
            r = run(fn, p["input"])
            if r is None or H(r) != H(o) or W(r) != W(o): preds = None; break
            if not preds and fit_cmap([r], [o]) is None: preds = None; break
            preds.append(r)
        if not preds: continue
        if preds == outs:
            found.append((name, cost, fn))
        else:
            m = fit_cmap(preds, outs)
            if m and all(apply_cmap(a, m) == b for a, b in zip(preds, outs)):
                found.append((name + "+colour-map", cost + 1, lambda g, fn=fn, m=m: apply_cmap(run(fn, g), m)))
        if len(found) >= max_programs: break
    res, seen = [], set()
    for name, cost, fn in sorted(found, key=lambda x: x[1]):
        preds = [run(fn, t["input"]) for t in task["test"]]
        if any(p is None for p in preds): continue
        k = str(preds)
        if k in seen: continue
        seen.add(k); res.append({"program": name, "cost": cost, "preds": preds})
    if not res and allow2:
        res = search2(task)
    return res

# ------------------------------------------------------------------ cycle 12: own primitives (latent/)
import importlib as _il
for _m in ("fam_lines", "fam_move_by_vector", "fam_crop_by_score", "fam_summarise_ranked_bars",
           "fam_move_slide_until_contact", "fam_fill_region_by_property", "fam_stamp_local_stencil"):
    try:
        FAMILIES = FAMILIES + tuple(_il.import_module(_m).FAMILIES)
    except Exception:
        pass


In [ ]:
%%writefile /kaggle/working/predict_m1.py
"""Kaggle driver for the M1b lattice/RDR probe (occupancy2.py).

Every prediction comes from rules induced on the task's own training pairs over the frozen prior
vocabulary; unresolved slots get a 1x1 [[0]] placeholder. Deterministic (sorted iteration), so a
local run and the Kaggle run over the same file must produce the same submission digest.
"""
import argparse, hashlib, json, os, signal, sys, time

PLACEHOLDER = [[0]]


class TaskTimeout(BaseException):
    pass


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--candidate", required=True)      # dir containing image.py / ARCGraph.py
    ap.add_argument("--probe", required=True)          # dir containing occupancy2.py
    ap.add_argument("--vocab", required=True)
    ap.add_argument("--challenges", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--task-seconds", type=int, default=120)
    args = ap.parse_args()
    os.environ["M1B_VOCAB"] = args.vocab
    sys.path[:0] = [args.probe, args.candidate]
    import occupancy2 as probe
    probe.Timeout = TaskTimeout
    ch = json.load(open(args.challenges))
    signal.signal(signal.SIGALRM, lambda *a: (_ for _ in ()).throw(TaskTimeout()))
    sub, solved_slots = {}, 0
    for k in sorted(ch):
        task = ch[k]
        slots = [{"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER} for _ in task["test"]]
        try:
            signal.alarm(args.task_seconds)
            attempts, _, _ = probe.solve(task)
            for rank, a in enumerate(attempts[:2]):
                for i, g in enumerate(a["preds"]):
                    if g and all(isinstance(r, list) and r for r in g):
                        slots[i]["attempt_1" if rank == 0 else "attempt_2"] = g
        except TaskTimeout:
            pass
        except Exception:
            pass
        finally:
            signal.alarm(0)
        sub[k] = slots
    blob = json.dumps(sub, sort_keys=True).encode()
    open(args.out, "w").write(json.dumps(sub))
    print(json.dumps({"tasks": len(sub), "digest": hashlib.sha256(blob).hexdigest()}))


if __name__ == "__main__":
    main()


In [ ]:
# Parity gate: the notebook must replicate the local result on the public 120-task evaluation set.
PROBE_SHA = '68f1332ce4f972a676f027ba9519bdf53cd1bb054d07f57a289c214887af87f2'
VOCAB = 'fill_interior,ray,connect,border,bbox_fill,outline8,L_inertia,D12,S_holes,D4,D1,S_size,P_pixel,P_frame,D10,S_atoms,G_dsl'
EVAL_SHA = 'e7c62a4bd211867c6b538f66b8013b81f299663c82ca062f49a52bf439d6e4e8'
EXPECTED_DIGEST = 'fee33f1f3d91f732f9c2b27ef4860a58f62048eae953bf118823fd5ed0f404c0'
EXPECTED_CORRECT = 9
assert hashlib.sha256(Path('/kaggle/working/probe/occupancy2.py').read_bytes()).hexdigest() == PROBE_SHA
evals = [p for p in Path('/kaggle/input').rglob('arc-agi_evaluation_challenges.json')
         if hashlib.sha256(p.read_bytes()).hexdigest() == EVAL_SHA]
assert evals, 'identical public evaluation file not found; parity cannot be verified'
sols = list(evals[0].parent.glob('arc-agi_evaluation_solutions.json'))
env = dict(os.environ, PYTHONHASHSEED='0', OMP_NUM_THREADS='1', MPLBACKEND='Agg',
           PYTHONPATH=os.pathsep.join([str(deps), os.environ.get('PYTHONPATH', '')]))
def run(ch, out):
    subprocess.run([sys.executable, '/kaggle/working/predict_m1.py', '--candidate', str(cand),
                    '--probe', '/kaggle/working/probe', '--vocab', VOCAB, '--challenges', str(ch),
                    '--out', out, '--task-seconds', '300'], check=True, env=env)
    return json.load(open(out))
ev = run(evals[0], '/kaggle/working/parity_eval.json')
digest = hashlib.sha256(json.dumps(ev, sort_keys=True).encode()).hexdigest()
correct = None
if sols:
    ans = json.load(open(sols[0]))
    correct = sum(any(ev[k][i][a] == t for a in ('attempt_1', 'attempt_2')) for k in ans for i, t in enumerate(ans[k]))
report = {'eval_file': str(evals[0]), 'prediction_digest': digest, 'digest_match': digest == EXPECTED_DIGEST,
          'correct_of_172': correct, 'expected_correct': EXPECTED_CORRECT}
json.dump(report, open('/kaggle/working/parity_report.json', 'w'), indent=1)
print(report)
assert digest == EXPECTED_DIGEST, 'PARITY FAILED: predictions differ from the frozen local run'
assert correct in (None, EXPECTED_CORRECT), f'PARITY FAILED: expected {EXPECTED_CORRECT} correct, got {correct}'


In [ ]:
sub = run(challenges[0], '/kaggle/working/submission.json')
src = json.load(open(challenges[0]))
assert set(sub) == set(src) and all(len(sub[k]) == len(src[k]['test']) for k in src)
print(len(sub), 'tasks written')
